In [1]:

from pathlib import Path
import duckdb
import pandas as pd

# Locate the repository root, whether the notebook runs from
# the notebooks/ directory or the repository root.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

# Canonical data lives here; A02 will eventually be written
# to data/analytical/, separately from the canonical sources.
canonical_dir = project_root / "data" / "canonical"

con = duckdb.connect()

# Confirm the files and directories before proceeding.
print("Project root:", project_root)
print("Canonical directory:", canonical_dir)
print("Canonical directory exists:", canonical_dir.is_dir())

for filename in ["player_team_seasons.csv", "player_salaries.csv"]:
    path = canonical_dir / filename
    print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")


Project root: /Users/jonathantran/Desktop/nba_economics
Canonical directory: /Users/jonathantran/Desktop/nba_economics/data/canonical
Canonical directory exists: True
player_team_seasons.csv: FOUND
player_salaries.csv: FOUND


In [2]:

sources = {
    "C05": canonical_dir / "player_team_seasons.csv",
    "C07": canonical_dir / "player_salaries.csv",
}

for name, path in sources.items():

    # Convert the Path object into a SQL-safe string literal.
    # This is appropriate here because these paths are defined
    # locally in our code, not supplied by external users.
    sql_path = str(path).replace("'", "''")

    # Create a temporary DuckDB view for each canonical CSV.
    # We embed the escaped file path directly because CREATE VIEW
    # cannot use the ? parameter placeholder in this context.
    con.execute(
        f"""
        CREATE OR REPLACE TEMP VIEW {name} AS
        SELECT *
        FROM read_csv_auto('{sql_path}')
        """
    )

    print(f"\n{'=' * 50}")
    print(f"{name} — {path.name}")

    # Inspect the column names and inferred data types.
    print("\nSCHEMA:")
    print(con.execute(f"DESCRIBE {name}").df().to_string(index=False))

    # Count the number of records.
    print("\nROW COUNT:")
    print(con.execute(f"SELECT COUNT(*) AS n_rows FROM {name}").df())

    # Preview the first three observations.
    print("\nSAMPLE:")
    print(con.execute(f"SELECT * FROM {name} LIMIT 3").df().to_string(index=False))



C05 — player_team_seasons.csv

SCHEMA:
                      column_name column_type null  key default extra
                        player_id      BIGINT  YES None    None  None
                     franchise_id      BIGINT  YES None    None  None
                           season     VARCHAR  YES None    None  None
                              age      DOUBLE  YES None    None  None
                    height_inches      DOUBLE  YES None    None  None
                       weight_lbs     VARCHAR  YES None    None  None
                     games_played      BIGINT  YES None    None  None
                     points_total      BIGINT  YES None    None  None
                   rebounds_total      BIGINT  YES None    None  None
                    assists_total      BIGINT  YES None    None  None
                             wins      BIGINT  YES None    None  None
                           losses      BIGINT  YES None    None  None
                          win_pct      DOUBLE  YES

In [3]:

# ------------------------------------------------------------
# A02: Structural validation of C05 and C07
# ------------------------------------------------------------

# Check whether either canonical source contains duplicate
# player-franchise-season keys.
for source in ["C05", "C07"]:

    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT (player_id, franchise_id, season))
                AS distinct_keys,
            COUNT(*) - COUNT(DISTINCT (player_id, franchise_id, season))
                AS duplicate_key_rows,
            COUNT(*) FILTER (
                WHERE player_id IS NULL
                   OR franchise_id IS NULL
                   OR season IS NULL
            ) AS null_key_rows
        FROM {source}
    """).df()

    print(f"\n{source} — KEY VALIDATION")
    print(result.to_string(index=False))


# Check whether the same player is associated with more than
# one recorded franchise during a single season.
multi_franchise = con.execute("""
    SELECT
        player_id,
        season,
        COUNT(DISTINCT franchise_id) AS franchise_count
    FROM C05
    GROUP BY player_id, season
    HAVING COUNT(DISTINCT franchise_id) > 1
    ORDER BY franchise_count DESC, season, player_id
""").df()

print("\nC05 — MULTIPLE FRANCHISES PER PLAYER-SEASON")
print("Affected player-seasons:", len(multi_franchise))
print(multi_franchise.head(10).to_string(index=False))


# Check C05-to-C07 join cardinality and salary coverage.
# LEFT JOIN preserves all C05 observations unless C07
# contains multiple matching salary records.
join_audit = con.execute("""
    SELECT
        COUNT(*) AS joined_rows,
        COUNT(*) FILTER (
            WHERE s.player_id IS NOT NULL
        ) AS matched_rows,
        COUNT(*) FILTER (
            WHERE s.player_id IS NULL
        ) AS unmatched_rows,
        COUNT(*) FILTER (
            WHERE s.salary IS NOT NULL
        ) AS salary_nonnull_rows
    FROM C05 AS p
    LEFT JOIN C07 AS s
        ON p.player_id = s.player_id
       AND p.franchise_id = s.franchise_id
       AND p.season = s.season
""").df()

print("\nC05 → C07 — JOIN AUDIT")
print(join_audit.to_string(index=False))



C05 — KEY VALIDATION
 total_rows  distinct_keys  duplicate_key_rows  null_key_rows
      12810          12810                   0              0

C07 — KEY VALIDATION
 total_rows  distinct_keys  duplicate_key_rows  null_key_rows
      11826          11826                   0              0

C05 — MULTIPLE FRANCHISES PER PLAYER-SEASON
Affected player-seasons: 0
Empty DataFrame
Columns: [player_id, season, franchise_count]
Index: []

C05 → C07 — JOIN AUDIT
 joined_rows  matched_rows  unmatched_rows  salary_nonnull_rows
       12810         10601            2209                10601


In [4]:

# ------------------------------------------------------------
# A02: Inspect essential player and season dimensions
# ------------------------------------------------------------

additional_sources = {
    "C03": canonical_dir / "players.csv",
    "C01": canonical_dir / "seasons.csv",
}

for name, path in additional_sources.items():

    # Ensure the canonical file exists before reading it.
    if not path.is_file():
        raise FileNotFoundError(f"Missing canonical source: {path}")

    # Escape single quotes so the local file path can be
    # safely embedded in the DuckDB SQL statement.
    sql_path = str(path).replace("'", "''")

    # Create a temporary view; canonical CSVs remain unchanged.
    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW {name} AS
        SELECT *
        FROM read_csv_auto('{sql_path}')
    """)

    print(f"\n{'=' * 55}")
    print(f"{name} — {path.name}")

    # Display all available columns and inferred types.
    print("\nSCHEMA:")
    print(con.execute(f"DESCRIBE {name}").df().to_string(index=False))

    # Confirm record counts.
    print("\nROW COUNT:")
    print(
        con.execute(
            f"SELECT COUNT(*) AS n_rows FROM {name}"
        ).df().to_string(index=False)
    )

    # Preview representative source records.
    print("\nSAMPLE:")
    print(
        con.execute(
            f"SELECT * FROM {name} LIMIT 3"
        ).df().to_string(index=False)
    )



C03 — players.csv

SCHEMA:
 column_name column_type null  key default extra
   player_id      BIGINT  YES None    None  None
 player_name     VARCHAR  YES None    None  None
     country     VARCHAR  YES None    None  None
     college     VARCHAR  YES None    None  None
  draft_year     VARCHAR  YES None    None  None
 draft_round     VARCHAR  YES None    None  None
draft_number     VARCHAR  YES None    None  None

ROW COUNT:
 n_rows
   2584

SAMPLE:
 player_id     player_name country          college draft_year draft_round draft_number
         3      Grant Long     USA Eastern Michigan       1988           2           33
        15 Eric Piatkowski     USA         Nebraska       1994           1           15
        21    Greg Anthony     USA             UNLV       1991           1           12

C01 — seasons.csv

SCHEMA:
         column_name column_type null  key default extra
              season     VARCHAR  YES None    None  None
                Year      BIGINT  YES None    Non

In [5]:

# ------------------------------------------------------------
# A02: Validate player and season dimensions
# ------------------------------------------------------------

# Each dimension should have one record per join key.
for source, key in [
    ("C03", "player_id"),
    ("C01", "season")
]:
    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT {key}) AS unique_keys,
            COUNT(*) - COUNT(DISTINCT {key}) AS duplicate_keys,
            COUNT(*) FILTER (WHERE {key} IS NULL) AS null_keys
        FROM {source}
    """).df()

    print(f"\n{source} — DIMENSION KEY VALIDATION")
    print(result.to_string(index=False))


# Check whether every C05 player has a corresponding
# player record in C03, without discarding unmatched rows.
player_coverage = con.execute("""
    SELECT
        COUNT(*) AS c05_rows,
        COUNT(*) FILTER (
            WHERE d.player_id IS NOT NULL
        ) AS matched_players,
        COUNT(*) FILTER (
            WHERE d.player_id IS NULL
        ) AS unmatched_players
    FROM C05 AS p
    LEFT JOIN C03 AS d
        ON p.player_id = d.player_id
""").df()

print("\nC05 → C03 — PLAYER JOIN")
print(player_coverage.to_string(index=False))


# Check whether every C05 season maps to C01.
season_coverage = con.execute("""
    SELECT
        COUNT(*) AS c05_rows,
        COUNT(*) FILTER (
            WHERE s.season IS NOT NULL
        ) AS matched_seasons,
        COUNT(*) FILTER (
            WHERE s.season IS NULL
        ) AS unmatched_seasons
    FROM C05 AS p
    LEFT JOIN C01 AS s
        ON p.season = s.season
""").df()

print("\nC05 → C01 — SEASON JOIN")
print(season_coverage.to_string(index=False))



C03 — DIMENSION KEY VALIDATION
 total_rows  unique_keys  duplicate_keys  null_keys
       2584         2584               0          0

C01 — DIMENSION KEY VALIDATION
 total_rows  unique_keys  duplicate_keys  null_keys
         26           26               0          0

C05 → C03 — PLAYER JOIN
 c05_rows  matched_players  unmatched_players
    12810            12810                  0

C05 → C01 — SEASON JOIN
 c05_rows  matched_seasons  unmatched_seasons
    12810            12810                  0


In [6]:

# ------------------------------------------------------------
# A02 v0: Player–franchise–season analytical base
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v0_base AS

    SELECT
        -- Preserve the complete player-performance record.
        p.*,

        -- C03: Player identity and draft background.
        d.player_name,
        d.country,
        d.college,
        d.draft_year,
        d.draft_round,
        d.draft_number,

        -- C07: Salary and explicit availability indicator.
        s.salary,
        CASE
            WHEN s.salary IS NOT NULL THEN 1
            ELSE 0
        END AS salary_available,

        -- C01: Financial context for the relevant season.
        c."Salary Cap" AS salary_cap,
        c."Luxury Tax" AS luxury_tax_threshold,
        c."1st Apron" AS first_apron_threshold,
        c."2nd Apron" AS second_apron_threshold,
        c.cba_id

    FROM C05 AS p

    LEFT JOIN C03 AS d
        ON p.player_id = d.player_id

    LEFT JOIN C07 AS s
        ON p.player_id = s.player_id
       AND p.franchise_id = s.franchise_id
       AND p.season = s.season

    LEFT JOIN C01 AS c
        ON p.season = c.season
""")

print("A02 v0 created successfully.")

# Check the size of the new analytical view.
print("\nA02 v0 DIMENSIONS:")
print(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(*) FILTER (WHERE salary_available = 1)
            AS salary_matched,
        COUNT(*) FILTER (WHERE salary_available = 0)
            AS salary_unmatched
    FROM a02_v0_base
""").df().to_string(index=False))

# Inspect a small selection of columns to verify the joins.
print("\nA02 v0 SAMPLE:")
print(con.execute("""
    SELECT
        player_name,
        season,
        franchise_id,
        games_played,
        points_per_game,
        salary,
        salary_cap,
        salary_available
    FROM a02_v0_base
    ORDER BY season, player_name
    LIMIT 10
""").df().to_string(index=False))


A02 v0 created successfully.

A02 v0 DIMENSIONS:
 rows  unique_keys  salary_matched  salary_unmatched
12810        12810           10601              2209

A02 v0 SAMPLE:
   player_name    season  franchise_id  games_played  points_per_game  salary  salary_cap  salary_available
    A.C. Green 2000-2001    1610612748            82              4.5 2250000    35500000                 1
   A.J. Guyton 2000-2001    1610612741            33              6.0  316000    35500000                 1
   Aaron McKie 2000-2001    1610612755            76             11.6 1820000    35500000                 1
Aaron Williams 2000-2001    1610612751            82             10.2 2250000    35500000                 1
    Adam Keefe 2000-2001    1610612744            67              2.5 3390000    35500000                 1
  Adonal Foyle 2000-2001    1610612744            58              5.9 3578000    35500000                 1
Adrian Griffin 2000-2001    1610612738            44              2.1  49

In [7]:

# ------------------------------------------------------------
# A02 v1 — Pass A: Salary normalization and comparisons
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v1_compensation_a AS

    WITH salary_benchmarks AS (
        SELECT
            season,
            franchise_id,

            -- League benchmarks repeat across franchise groups,
            -- so calculate league-wide values with window functions.
            AVG(salary) OVER (
                PARTITION BY season
            ) AS league_mean_salary,

            MEDIAN(salary) OVER (
                PARTITION BY season
            ) AS league_median_salary,

            -- Franchise-level benchmarks.
            AVG(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS team_mean_salary,

            MEDIAN(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS team_median_salary,

            player_id,
            salary

        FROM a02_v0_base
    ),

    salary_features AS (
        SELECT
            p.*,

            -- Historical normalization against the salary cap.
            p.salary * 1.0 / NULLIF(p.salary_cap, 0)
                AS salary_to_cap_ratio,

            -- Rank within the recorded franchise-season.
            CASE
                WHEN p.salary IS NOT NULL THEN
                    RANK() OVER (
                        PARTITION BY p.franchise_id, p.season
                        ORDER BY p.salary DESC NULLS LAST
                    )
            END AS salary_rank_franchise_season,

            -- Rank across the entire league in the same season.
            CASE
                WHEN p.salary IS NOT NULL THEN
                    RANK() OVER (
                        PARTITION BY p.season
                        ORDER BY p.salary DESC NULLS LAST
                    )
            END AS salary_rank_league_season,

            -- Salary percentile among matched league salaries.
            CASE
                WHEN p.salary IS NOT NULL THEN
                    (
                        RANK() OVER (
                            PARTITION BY p.season
                            ORDER BY p.salary ASC NULLS LAST
                        ) - 1
                    ) * 1.0
                    / NULLIF(
                        COUNT(p.salary) OVER (
                            PARTITION BY p.season
                        ) - 1,
                        0
                    )
            END AS salary_percentile_league_season

        FROM a02_v0_base AS p
    )

    SELECT
        f.*,

        -- Reference statistics retained for interpretability.
        b.league_mean_salary,
        b.league_median_salary,
        b.team_mean_salary,
        b.team_median_salary,

        -- Player salary relative to each benchmark.
        f.salary * 1.0 / NULLIF(b.league_mean_salary, 0)
            AS salary_vs_league_mean,

        f.salary * 1.0 / NULLIF(b.league_median_salary, 0)
            AS salary_vs_league_median,

        f.salary * 1.0 / NULLIF(b.team_mean_salary, 0)
            AS salary_vs_team_mean,

        f.salary * 1.0 / NULLIF(b.team_median_salary, 0)
            AS salary_vs_team_median

    FROM salary_features AS f

    LEFT JOIN salary_benchmarks AS b
        ON f.player_id = b.player_id
       AND f.franchise_id = b.franchise_id
       AND f.season = b.season
""")

print("A02 v1 Pass A created successfully.")

# ------------------------------------------------------------
# Validate that no player-season records were duplicated.
# ------------------------------------------------------------

result = con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(salary_to_cap_ratio) AS cap_ratios,
        COUNT(salary_rank_league_season) AS league_ranks,
        COUNT(salary_vs_league_median) AS league_median_ratios,
        COUNT(salary_vs_team_median) AS team_median_ratios
    FROM a02_v1_compensation_a
""").df()

print("\nA02 v1 PASS A VALIDATION:")
print(result.to_string(index=False))


A02 v1 Pass A created successfully.

A02 v1 PASS A VALIDATION:
 rows  unique_keys  cap_ratios  league_ranks  league_median_ratios  team_median_ratios
12810        12810       10601         10601                 10601               10601


In [8]:
# ------------------------------------------------------------
# A02 v1 Pass A — Semantic validation
# ------------------------------------------------------------

# Inspect the league's highest-paid players in a recent season.
top_salaries = con.execute("""
    SELECT
        player_name,
        season,
        salary,
        salary_rank_league_season,
        salary_rank_franchise_season,
        ROUND(salary_to_cap_ratio, 3) AS salary_to_cap,
        ROUND(salary_percentile_league_season, 3) AS league_percentile,
        ROUND(salary_vs_league_median, 2) AS vs_league_median,
        ROUND(salary_vs_team_median, 2) AS vs_team_median
    FROM a02_v1_compensation_a
    WHERE season = '2024-2025'
      AND salary IS NOT NULL
    ORDER BY salary DESC
    LIMIT 10
""").df()

print("TOP SALARIES — 2024-2025")
print(top_salaries.to_string(index=False))


# Check for invalid ratios, rankings, and percentiles.
validation = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE salary_to_cap_ratio < 0
        ) AS negative_cap_ratios,

        COUNT(*) FILTER (
            WHERE salary_percentile_league_season < 0
               OR salary_percentile_league_season > 1
        ) AS invalid_percentiles,

        COUNT(*) FILTER (
            WHERE salary IS NOT NULL
              AND salary_rank_league_season < 1
        ) AS invalid_league_ranks,

        COUNT(*) FILTER (
            WHERE salary IS NOT NULL
              AND salary_rank_franchise_season < 1
        ) AS invalid_team_ranks,

        COUNT(*) FILTER (
            WHERE salary IS NULL
              AND (
                  salary_rank_league_season IS NOT NULL
                  OR salary_percentile_league_season IS NOT NULL
              )
        ) AS unmatched_salary_with_rank

    FROM a02_v1_compensation_a
""").df()

print("\nFEATURE VALIDATION")
print(validation.to_string(index=False))

TOP SALARIES — 2024-2025
       player_name    season   salary  salary_rank_league_season  salary_rank_franchise_season  salary_to_cap  league_percentile  vs_league_median  vs_team_median
     Stephen Curry 2024-2025 55761216                          1                             1          0.397              1.000             10.79            8.30
      Nikola Jokić 2024-2025 51415938                          2                             1          0.366              0.996              9.95           16.08
       Joel Embiid 2024-2025 51415938                          2                             1          0.366              0.996              9.95           18.69
      Kevin Durant 2024-2025 51179021                          4                             1          0.364              0.993              9.90           10.24
      Bradley Beal 2024-2025 50203930                          5                             2          0.357              0.991              9.71           10.

In [9]:
# ------------------------------------------------------------
# A02 v1 — Pass B: Salary allocation and distribution
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v1_compensation_b AS

    WITH salary_pools AS (
        SELECT
            *,

            -- Total successfully matched salaries for each
            -- recorded franchise-season.
            SUM(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS matched_roster_salary_total,

            -- Total matched salaries across the entire league
            -- for the same season.
            SUM(salary) OVER (
                PARTITION BY season
            ) AS matched_league_salary_total,

            -- Count of salary-matched players on each
            -- recorded franchise-season.
            COUNT(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS matched_roster_salary_count,

            -- Total player records for the franchise-season,
            -- including those without matched salaries.
            COUNT(*) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_roster_player_count

        FROM a02_v1_compensation_a
    )

    SELECT
        *,

        -- Share of the franchise's matched salary pool.
        salary * 1.0 /
            NULLIF(matched_roster_salary_total, 0)
            AS salary_share_recorded_roster,

        -- Share of all matched league salaries that season.
        salary * 1.0 /
            NULLIF(matched_league_salary_total, 0)
            AS salary_share_league,

        -- Fraction of recorded franchise-season players
        -- with successfully matched salary records.
        matched_roster_salary_count * 1.0 /
            NULLIF(recorded_roster_player_count, 0)
            AS roster_salary_match_rate

    FROM salary_pools
""")

print("A02 v1 Pass B created successfully.")

print("\nA02 v1 PASS B VALIDATION:")

print(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(salary_share_recorded_roster)
            AS roster_shares,
        COUNT(salary_share_league)
            AS league_shares,
        MIN(roster_salary_match_rate)
            AS min_roster_match_rate,
        MAX(roster_salary_match_rate)
            AS max_roster_match_rate
    FROM a02_v1_compensation_b
""").df().to_string(index=False))

A02 v1 Pass B created successfully.

A02 v1 PASS B VALIDATION:
 rows  unique_keys  roster_shares  league_shares  min_roster_match_rate  max_roster_match_rate
12810        12810          10601          10601               0.066667                    1.0


In [10]:
# ------------------------------------------------------------
# A02 v1 — Identify low-coverage franchise-seasons
# ------------------------------------------------------------

low_coverage = con.execute("""
    SELECT
        season,
        franchise_id,
        MAX(recorded_roster_player_count) AS recorded_players,
        MAX(matched_roster_salary_count) AS matched_players,
        ROUND(MAX(roster_salary_match_rate), 3) AS match_rate,
        MAX(matched_roster_salary_total) AS matched_salary_total
    FROM a02_v1_compensation_b
    GROUP BY season, franchise_id
    ORDER BY match_rate ASC, season
    LIMIT 15
""").df()

print("LOWEST SALARY-MATCH COVERAGE:")
print(low_coverage.to_string(index=False))

LOWEST SALARY-MATCH COVERAGE:
   season  franchise_id  recorded_players  matched_players  match_rate  matched_salary_total
2004-2005    1610612765                15                1       0.067             6000000.0
2004-2005    1610612738                12                1       0.083            14625000.0
2004-2005    1610612746                17                2       0.118            12086000.0
2004-2005    1610612764                16                2       0.125             8404000.0
2004-2005    1610612762                16                2       0.125             8047000.0
2004-2005    1610612741                15                2       0.133            15150000.0
2004-2005    1610612754                20                3       0.150            15606000.0
2004-2005    1610612740                18                3       0.167             7864000.0
2003-2004    1610612762                17                3       0.176            21290000.0
2004-2005    1610612737                1

In [11]:
# ------------------------------------------------------------
# A02 v1 — Salary coverage by season
# ------------------------------------------------------------

season_coverage = con.execute("""
    SELECT
        season,

        -- Total player-franchise-season observations.
        COUNT(*) AS recorded_players,

        -- Observations with successfully matched salaries.
        COUNT(salary) AS matched_players,

        -- Share of observations with matched salaries.
        ROUND(
            COUNT(salary) * 1.0 / NULLIF(COUNT(*), 0),
            3
        ) AS salary_match_rate,

        -- Total observed salaries for this season.
        ROUND(SUM(salary) / 1000000.0, 2)
            AS matched_salary_total_millions

    FROM a02_v1_compensation_b

    GROUP BY season
    ORDER BY season
""").df()

print("SALARY COVERAGE BY SEASON:")
print(season_coverage.to_string(index=False))

SALARY COVERAGE BY SEASON:
   season  recorded_players  matched_players  salary_match_rate  matched_salary_total_millions
2000-2001               441              369              0.837                        1293.90
2001-2002               440              411              0.934                        1398.28
2002-2003               428              278              0.650                        1321.13
2003-2004               442              177              0.400                        1040.09
2004-2005               464              118              0.254                         813.44
2005-2006               458              420              0.917                        1696.24
2006-2007               458              432              0.943                        1728.04
2007-2008               451              429              0.951                        1823.52
2008-2009               445              427              0.960                        2000.96
2009-2010              

In [12]:
# ------------------------------------------------------------
# A02 v1 — Final compensation view and coverage validation
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v1_compensation AS

    SELECT
        *,

        -- Number of salary-matched player observations
        -- across the league in the same season.
        COUNT(salary) OVER (
            PARTITION BY season
        ) AS matched_league_salary_count,

        -- Proportion of league player observations
        -- with matched salary records.
        COUNT(salary) OVER (
            PARTITION BY season
        ) * 1.0
        / NULLIF(
            COUNT(*) OVER (PARTITION BY season),
            0
        ) AS league_salary_match_rate

    FROM a02_v1_compensation_b
""")

# Each franchise-season's observed salary shares should
# sum to 1, provided at least one salary is matched.
share_validation = con.execute("""
    WITH team_shares AS (
        SELECT
            season,
            franchise_id,
            SUM(salary_share_recorded_roster)
                AS total_share
        FROM a02_v1_compensation
        GROUP BY season, franchise_id
    ),

    league_shares AS (
        SELECT
            season,
            SUM(salary_share_league)
                AS total_share
        FROM a02_v1_compensation
        GROUP BY season
    )

    SELECT
        (SELECT COUNT(*) FROM team_shares
         WHERE ABS(total_share - 1.0) > 0.000001)
            AS invalid_team_share_totals,

        (SELECT COUNT(*) FROM team_shares
         WHERE total_share IS NULL)
            AS teams_without_matched_salaries,

        (SELECT COUNT(*) FROM league_shares
         WHERE ABS(total_share - 1.0) > 0.000001)
            AS invalid_league_share_totals

""").df()

print("A02 v1 FINAL SHARE VALIDATION:")
print(share_validation.to_string(index=False))

A02 v1 FINAL SHARE VALIDATION:
 invalid_team_share_totals  teams_without_matched_salaries  invalid_league_share_totals
                         0                               0                            0


In [13]:
from pathlib import Path

# ------------------------------------------------------------
# Step 1 — Locate the existing A01 analytical Parquet file
# ------------------------------------------------------------

# Path is relative to your notebook's working directory.
analytical_dir = Path("../data/analytical")

# Find Parquet files whose names begin with "a01".
a01_files = sorted(analytical_dir.glob("a01*.parquet"))

print("A01 PARQUET FILES FOUND:")

for file in a01_files:
    print(file)

# ------------------------------------------------------------
# Step 2 — Inspect payroll-related columns
# ------------------------------------------------------------

if len(a01_files) == 1:

    # DuckDB can query Parquet directly from its file path.
    # No permanent table or temporary view is required.
    a01_path = str(a01_files[0])

    a01_columns = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [a01_path]
    ).df()

    payroll_candidates = a01_columns[
        a01_columns["column_name"].str.contains(
            "payroll|salary|taxable|cap_space|dead_money",
            case=False,
            regex=True
        )
    ]

    print("\nPOTENTIAL PAYROLL DENOMINATORS:")
    print(
        payroll_candidates[
            ["column_name", "column_type"]
        ].to_string(index=False)
    )

elif len(a01_files) == 0:
    print("No matching Parquet file found. Check the directory path.")

else:
    print("Multiple A01 files found. Select the intended analytical table.")

A01 PARQUET FILES FOUND:
../data/analytical/a01_franchise_season.parquet

POTENTIAL PAYROLL DENOMINATORS:
                                  column_name column_type
                                 team_payroll      BIGINT
                               salary_cap_hit      BIGINT
                             salary_cap_space      DOUBLE
                              taxable_payroll      DOUBLE
                                active_salary      DOUBLE
                                  dead_salary      DOUBLE
                                   salary_cap      BIGINT
                         salary_cap_floor_pct      DOUBLE
                             salary_cap_floor      BIGINT
                         payroll_to_cap_ratio      DOUBLE
                 taxable_payroll_to_tax_ratio      DOUBLE
                    tax_paid_to_payroll_ratio      DOUBLE
              estimated_cost_to_taxable_ratio      DOUBLE
                              payroll_per_win      DOUBLE
                      ta

In [14]:
# ------------------------------------------------------------
# A02 v1 — Register A01 as a DuckDB reference view
# ------------------------------------------------------------

# a01_path was established in Cell 13.
# We use a Python f-string to insert the verified file path
# directly into the SQL definition.
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW a01_reference AS
    SELECT *
    FROM read_parquet('{a01_path}')
""")

print("A01 reference view created successfully.")

# Compare one observation per franchise-season.
# MAX() is appropriate because the roster-level metrics
# repeat across every player in that franchise-season.
comparison = con.execute("""
    WITH a02_roster_totals AS (
        SELECT
            franchise_id,
            season,
            MAX(matched_roster_salary_total)
                AS matched_salary_total,
            MAX(roster_salary_match_rate)
                AS salary_match_rate,
            MAX(recorded_roster_player_count)
                AS recorded_players

        FROM a02_v1_compensation
        GROUP BY franchise_id, season
    )

    SELECT
        p.season,
        p.franchise_id,
        p.recorded_players,
        ROUND(p.salary_match_rate, 3)
            AS salary_match_rate,

        ROUND(p.matched_salary_total / 1000000.0, 2)
            AS matched_salary_millions,

        ROUND(a.active_salary / 1000000.0, 2)
            AS active_salary_millions,

        ROUND(
            p.matched_salary_total * 1.0
            / NULLIF(a.active_salary, 0),
            3
        ) AS matched_to_active_ratio

    FROM a02_roster_totals AS p

    LEFT JOIN a01_reference AS a
        ON p.franchise_id = a.franchise_id
       AND p.season = a.season

    ORDER BY p.season DESC, p.franchise_id
""").df()

print("A02 vs A01 ACTIVE SALARY COMPARISON:")
print(comparison.head(20).to_string(index=False))

print("\nMATCHED-TO-ACTIVE RATIO SUMMARY:")
print(
    comparison["matched_to_active_ratio"]
    .describe()
    .round(3)
    .to_string()
)

print("\nMISSING A01 ACTIVE SALARY VALUES:")
print(comparison["active_salary_millions"].isna().sum())

A01 reference view created successfully.
A02 vs A01 ACTIVE SALARY COMPARISON:
   season  franchise_id  recorded_players  salary_match_rate  matched_salary_millions  active_salary_millions  matched_to_active_ratio
2025-2026    1610612737                21              0.762                   181.73                  180.37                    1.008
2025-2026    1610612738                17              0.765                   189.35                  187.42                    1.010
2025-2026    1610612739                22              0.727                   221.98                  212.07                    1.047
2025-2026    1610612740                18              0.833                   182.15                  182.26                    0.999
2025-2026    1610612741                20              0.750                   176.22                  166.17                    1.060
2025-2026    1610612742                19              0.789                   139.01                  176.20   

In [15]:
from pathlib import Path
import json

# Start from the repository root.
repo_root = Path("..")

# Search terms relevant to the S08 financial-data lineage.
keywords = ["active_salary", "s08", "hoopshype"]

# Track files that cannot be read as valid notebook JSON.
skipped_files = []

for notebook_path in repo_root.rglob("*.ipynb"):

    try:
        # Open the notebook and parse its JSON structure.
        with open(notebook_path, encoding="utf-8") as f:
            notebook = json.load(f)

    except (json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
        # Record the problematic file and continue searching.
        skipped_files.append((str(notebook_path), str(e)))
        continue

    # Examine code cells only, excluding notebook outputs.
    for cell_number, cell in enumerate(notebook.get("cells", []), 1):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        # Find lines containing any of our search terms.
        matches = [
            i for i, line in enumerate(lines)
            if any(word in line.lower() for word in keywords)
        ]

        if not matches:
            continue

        print(f"\nNOTEBOOK: {notebook_path}")
        print(f"CELL: {cell_number}")

        # Print three lines before and after each matching line.
        shown = set()

        for i in matches:
            for j in range(max(0, i - 3), min(len(lines), i + 4)):
                if j not in shown:
                    print(f"{j + 1:>3}: {lines[j]}")
                    shown.add(j)

# Report any notebooks that couldn't be inspected.
print("\n--- SKIPPED NOTEBOOKS ---")

if skipped_files:
    for path, error in skipped_files:
        print(f"{path}: {error}")
else:
    print("None")


NOTEBOOK: ../notebooks/18_s18_luxury_tax.ipynb
CELL: 6
 11:     "Space": "tax_space",
 12:     "Tax BillEstimated": "estimated_tax_bill",
 13:     "TotalTax Payroll + Est. Bill": "total_estimated_cost",
 14:     "Active": "active_salary",
 15:     "DeadEstimated": "dead_salary"
 16: })
 17: 

NOTEBOOK: ../notebooks/18_s18_luxury_tax.ipynb
CELL: 7
  4:     "tax_space",
  5:     "estimated_tax_bill",
  6:     "total_estimated_cost",
  7:     "active_salary",
  8:     "dead_salary"
  9: ]
 10: 

NOTEBOOK: ../notebooks/18_s18_luxury_tax.ipynb
CELL: 13
 12:     ).abs() > 1,
 13: 
 14:     "salary_breakdown_mismatch": (
 15:         recent_clean["active_salary"]
 16:         + recent_clean["dead_salary"].fillna(0)
 17:         - recent_clean["taxable_payroll"]
 18:     ).abs() > 1

NOTEBOOK: ../notebooks/18_s18_luxury_tax.ipynb
CELL: 14
  5:             "franchise_id",
  6:             "season",
  7:             "taxable_payroll",
  8:             "active_salary",
  9:             "dead_sal

In [16]:

# ------------------------------------------------------------
# A02 v1 — ESPN vs Spotrac: targeted salary reconciliation
# ------------------------------------------------------------

salary_reconciliation = con.execute("""
    WITH team_totals AS (
        SELECT
            franchise_id,
            season,

            -- Each of these values repeats across the
            -- franchise's player records, so use MAX().
            MAX(matched_roster_salary_total) AS espn_salary_total,
            MAX(roster_salary_match_rate) AS match_rate,
            MAX(matched_roster_salary_count) AS matched_players,
            MAX(recorded_roster_player_count) AS recorded_players

        FROM a02_v1_compensation
        GROUP BY franchise_id, season
    )

    SELECT
        t.season,
        t.franchise_id,
        t.recorded_players,
        t.matched_players,
        ROUND(t.match_rate, 3) AS match_rate,

        ROUND(t.espn_salary_total / 1e6, 2)
            AS espn_salary_millions,

        ROUND(a.active_salary / 1e6, 2)
            AS spotrac_active_millions,

        -- Positive means ESPN exceeds Spotrac.
        ROUND(
            (t.espn_salary_total - a.active_salary) / 1e6,
            2
        ) AS difference_millions,

        -- Ratio of ESPN's matched total to Spotrac's active total.
        ROUND(
            t.espn_salary_total / NULLIF(a.active_salary, 0),
            3
        ) AS espn_to_spotrac_ratio

    FROM team_totals AS t

    LEFT JOIN a01_reference AS a
        ON t.franchise_id = a.franchise_id
       AND t.season = a.season

    WHERE t.season = '2024-2025'
    ORDER BY t.match_rate DESC, t.franchise_id
""").df()

print("2024-25 ESPN / SPOTRAC RECONCILIATION")
display(salary_reconciliation)

# Focus on teams with relatively complete ESPN salary matching.
high_coverage = salary_reconciliation[
    (salary_reconciliation["match_rate"] >= 0.90)
    & salary_reconciliation["espn_to_spotrac_ratio"].notna()
]

print("\nHIGH-COVERAGE FRANCHISES (90%+):")
print("Count:", len(high_coverage))

if not high_coverage.empty:
    print(
        high_coverage[
            ["match_rate", "espn_to_spotrac_ratio"]
        ].describe().round(3)
    )

    # Show the largest discrepancies even among high-coverage teams.
    display(
        high_coverage.assign(
            deviation=lambda df:
                (df["espn_to_spotrac_ratio"] - 1).abs()
        )
        .sort_values("deviation", ascending=False)
        .head(5)
    )


2024-25 ESPN / SPOTRAC RECONCILIATION


,season,franchise_id,recorded_players,matched_players,match_rate,espn_salary_millions,spotrac_active_millions,difference_millions,espn_to_spotrac_ratio
0,2024-2025,1610612757,19,17,0.895,169.18,165.52,3.66,1.022
1,2024-2025,1610612764,19,17,0.895,166.51,162.58,3.93,1.024
2,2024-2025,1610612745,18,16,0.889,163.40,160.43,2.97,1.019
3,2024-2025,1610612746,18,16,0.889,169.58,166.99,2.59,1.016
4,2024-2025,1610612752,18,16,0.889,190.74,188.00,2.73,1.015
5,2024-2025,1610612741,20,17,0.850,165.69,157.18,8.51,1.054
6,2024-2025,1610612750,20,17,0.850,205.46,198.65,6.81,1.034
7,2024-2025,1610612748,19,16,0.842,176.14,173.60,2.54,1.015
8,2024-2025,1610612737,18,15,0.833,161.26,161.26,0.00,1.000
9,2024-2025,1610612756,18,15,0.833,211.01,210.56,0.45,1.002



HIGH-COVERAGE FRANCHISES (90%+):
Count: 0


In [17]:

# ------------------------------------------------------------
# A02 v1 — Pass C: Cross-source salary comparison
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v1_final AS

    SELECT
        p.*,

        -- Spotrac team-level active salary, repeated for each
        -- player belonging to that franchise-season.
        a.active_salary AS spotrac_active_salary,

        -- Exploratory cross-source comparison:
        -- ESPN player salary / Spotrac active salary.
        p.salary * 1.0
            / NULLIF(a.active_salary, 0)
            AS salary_to_spotrac_active_ratio,

        -- Compare all matched ESPN salaries for the franchise
        -- against the Spotrac active-salary total.
        p.matched_roster_salary_total * 1.0
            / NULLIF(a.active_salary, 0)
            AS espn_to_spotrac_roster_ratio,

        -- Absolute relative difference between the two totals.
        ABS(
            p.matched_roster_salary_total - a.active_salary
        ) * 1.0 / NULLIF(a.active_salary, 0)
            AS espn_spotrac_relative_gap

    FROM a02_v1_compensation AS p

    LEFT JOIN a01_reference AS a
        ON p.franchise_id = a.franchise_id
       AND p.season = a.season
""")

# Confirm the new join did not change the analytical grain.
final_check = con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(salary_to_spotrac_active_ratio)
            AS cross_source_ratios,
        COUNT(*) FILTER (
            WHERE spotrac_active_salary IS NOT NULL
        ) AS rows_with_spotrac_salary
    FROM a02_v1_final
""").df()

display(final_check)


,rows,unique_keys,cross_source_ratios,rows_with_spotrac_salary
0,12810,12810,6681,7889


In [18]:

# ------------------------------------------------------------
# A02 v1 — F2: Player production source inventory
# ------------------------------------------------------------

# Retrieve column names and data types from the existing
# C05-based analytical view. No data is modified.
schema = con.execute("""
    DESCRIBE a02_v0_base
""").df()

# Find columns that could support competitive analysis.
# This is a discovery filter, not our final feature list.
keywords = [
    "game", "minute", "point", "rebound", "assist",
    "steal", "block", "turnover", "shoot",
    "field_goal", "three", "free_throw",
    "usage", "efficiency", "plus_minus", "win_share"
]

production_columns = schema[
    schema["column_name"].str.lower().apply(
        lambda name: any(word in name for word in keywords)
    )
]

print("A02 v0 total columns:", len(schema))
print("\nPOTENTIAL PRODUCTION COLUMNS:")
display(production_columns[["column_name", "column_type"]])

# Show the actual available production fields for one
# recent season, without guessing their names.
selected = production_columns["column_name"].tolist()

if selected:
    # Quote column names safely for DuckDB SQL.
    quoted = [
        '"' + name.replace('"', '""') + '"'
        for name in selected
    ]

    sample_columns = ", ".join(
        ['"player_id"', '"season"'] + quoted
    )

    sample = con.execute(f"""
        SELECT {sample_columns}
        FROM a02_v0_base
        WHERE season = '2024-2025'
        LIMIT 5
    """).df()

    print("\n2024-25 PLAYER SAMPLE:")
    display(sample)


A02 v0 total columns: 62

POTENTIAL PRODUCTION COLUMNS:


,column_name,column_type
6,games_played,BIGINT
7,points_total,BIGINT
8,rebounds_total,BIGINT
9,assists_total,BIGINT
13,minutes_per_game,DOUBLE
14,field_goals_made_per_game,DOUBLE
15,field_goals_attempted_per_game,DOUBLE
16,field_goal_pct,DOUBLE
17,three_pointers_made_per_game,DOUBLE
18,three_pointers_attempted_per_game,DOUBLE



2024-25 PLAYER SAMPLE:


,player_id,season,games_played,points_total,rebounds_total,assists_total,minutes_per_game,field_goals_made_per_game,field_goals_attempted_per_game,field_goal_pct,...,assist_pct,assist_to_turnover_ratio,assist_ratio,offensive_rebound_pct,defensive_rebound_pct,rebound_pct,team_turnover_pct,effective_field_goal_pct,true_shooting_pct,usage_pct
0,1630639,2024-2025,26,236,86,31,18.7,3.1,7.3,0.421,...,0.093,2.07,11.7,0.036,0.128,0.080,5.7,0.508,0.542,0.189
1,1631260,2024-2025,73,541,174,108,22.7,2.5,5.8,0.429,...,0.086,2.70,18.6,0.011,0.087,0.051,6.9,0.612,0.621,0.123
2,1642358,2024-2025,29,220,59,76,22.0,2.8,7.3,0.385,...,0.178,2.17,22.3,0.011,0.073,0.042,10.3,0.441,0.480,0.173
3,203932,2024-2025,51,748,247,164,28.4,5.2,9.7,0.531,...,0.143,2.25,20.1,0.060,0.109,0.086,9.0,0.607,0.650,0.186
4,1628988,2024-2025,62,340,78,83,12.8,1.9,4.3,0.437,...,0.142,2.24,20.5,0.015,0.073,0.044,9.1,0.571,0.594,0.162


In [19]:

# ------------------------------------------------------------
# A02 v1 — F2: Production and competitive contribution
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_f2_production AS

    WITH player_metrics AS (
        SELECT
            p.*,

            -- Estimate season minutes and defensive totals.
            -- Per-game source values may be rounded.
            games_played * minutes_per_game
                AS minutes_total_est,

            games_played * blocks_per_game
                AS blocks_total_est,

            games_played * steals_per_game
                AS steals_total_est

        FROM a02_v1_final AS p
    ),

    team_metrics AS (
        SELECT
            p.*,

            -- Recorded franchise-season totals.
            SUM(points_total) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_team_points_total,

            SUM(assists_total) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_team_assists_total,

            SUM(rebounds_total) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_team_rebounds_total,

            SUM(blocks_total_est) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_team_blocks_total_est,

            SUM(steals_total_est) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_team_steals_total_est,

            SUM(minutes_total_est) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_team_minutes_total_est,

            -- Rank players by scoring and offensive usage.
            RANK() OVER (
                PARTITION BY franchise_id, season
                ORDER BY points_per_game DESC NULLS LAST
            ) AS team_points_rank,

            RANK() OVER (
                PARTITION BY franchise_id, season
                ORDER BY usage_pct DESC NULLS LAST
            ) AS team_usage_rank,

            -- League-relative scoring percentile.
            -- Calculated over player-franchise-season records.
            PERCENT_RANK() OVER (
                PARTITION BY season
                ORDER BY points_per_game ASC NULLS FIRST
            ) AS league_points_percentile

        FROM player_metrics AS p
    )

    SELECT
        p.*,

        -- Production shares of recorded franchise totals.
        points_total * 1.0
            / NULLIF(recorded_team_points_total, 0)
            AS team_scoring_share,

        assists_total * 1.0
            / NULLIF(recorded_team_assists_total, 0)
            AS team_assist_share,

        rebounds_total * 1.0
            / NULLIF(recorded_team_rebounds_total, 0)
            AS team_rebound_share,

        blocks_total_est * 1.0
            / NULLIF(recorded_team_blocks_total_est, 0)
            AS team_blocks_share,

        steals_total_est * 1.0
            / NULLIF(recorded_team_steals_total_est, 0)
            AS team_steals_share,

        minutes_total_est * 1.0
            / NULLIF(recorded_team_minutes_total_est, 0)
            AS team_minutes_share,

        -- Exploratory scoring-to-turnover ratio.
        -- NULL when turnover rate is zero or unavailable.
        points_per_game * 1.0
            / NULLIF(turnovers_per_game, 0)
            AS points_per_turnover

    FROM team_metrics AS p
""")

print("A02 F2 production view created.")


A02 F2 production view created.


In [20]:

# ------------------------------------------------------------
# A02 F2 — Structural and coverage validation
# ------------------------------------------------------------

f2_check = con.execute("""
    SELECT
        COUNT(*) AS rows,

        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,

        COUNT(team_scoring_share)
            AS scoring_share_available,

        COUNT(team_assist_share)
            AS assist_share_available,

        COUNT(team_rebound_share)
            AS rebound_share_available,

        COUNT(team_blocks_share)
            AS blocks_share_available,

        COUNT(team_steals_share)
            AS steals_share_available,

        COUNT(team_minutes_share)
            AS minutes_share_available,

        COUNT(points_per_turnover)
            AS points_turnover_available

    FROM a02_f2_production
""").df()

display(f2_check)


,rows,unique_keys,scoring_share_available,assist_share_available,rebound_share_available,blocks_share_available,steals_share_available,minutes_share_available,points_turnover_available
0,12810,12810,12810,12810,12810,12810,12810,12810,12449


In [21]:

# ------------------------------------------------------------
# A02 F2 — Validate recorded roster contribution shares
# ------------------------------------------------------------

f2_share_check = con.execute("""
    WITH franchise_totals AS (
        SELECT
            franchise_id,
            season,

            SUM(team_scoring_share) AS scoring_sum,
            SUM(team_assist_share) AS assist_sum,
            SUM(team_rebound_share) AS rebound_sum,
            SUM(team_blocks_share) AS blocks_sum,
            SUM(team_steals_share) AS steals_sum,
            SUM(team_minutes_share) AS minutes_sum

        FROM a02_f2_production
        GROUP BY franchise_id, season
    )

    SELECT
        COUNT(*) AS franchise_seasons,

        COUNT(*) FILTER (
            WHERE ABS(scoring_sum - 1) > 0.0001
        ) AS scoring_failures,

        COUNT(*) FILTER (
            WHERE ABS(assist_sum - 1) > 0.0001
        ) AS assist_failures,

        COUNT(*) FILTER (
            WHERE ABS(rebound_sum - 1) > 0.0001
        ) AS rebound_failures,

        COUNT(*) FILTER (
            WHERE ABS(blocks_sum - 1) > 0.0001
        ) AS blocks_failures,

        COUNT(*) FILTER (
            WHERE ABS(steals_sum - 1) > 0.0001
        ) AS steals_failures,

        COUNT(*) FILTER (
            WHERE ABS(minutes_sum - 1) > 0.0001
        ) AS minutes_failures

    FROM franchise_totals
""").df()

display(f2_share_check)


,franchise_seasons,scoring_failures,assist_failures,rebound_failures,blocks_failures,steals_failures,minutes_failures
0,776,0,0,0,0,0,0


In [22]:

# Inspect existing compensation, ranking, and share columns.
# DuckDB DESCRIBE returns "column_type", not "data_type".

f3_columns = con.execute("""
    SELECT
        column_name,
        column_type
    FROM (
        DESCRIBE a02_f2_production
    )
    WHERE
        LOWER(column_name) LIKE '%salary%'
        OR LOWER(column_name) LIKE '%rank%'
        OR LOWER(column_name) LIKE '%share%'
        OR LOWER(column_name) LIKE '%usage%'
    ORDER BY column_name
""").df()

display(f3_columns)


,column_name,column_type
0,league_mean_salary,DOUBLE
1,league_median_salary,DOUBLE
2,league_salary_match_rate,DOUBLE
3,matched_league_salary_count,BIGINT
4,matched_league_salary_total,HUGEINT
5,matched_roster_salary_count,BIGINT
6,matched_roster_salary_total,HUGEINT
7,roster_salary_match_rate,DOUBLE
8,salary,BIGINT
9,salary_available,INTEGER


In [23]:

# ------------------------------------------------------------
# A02 v1 — F3: Compensation vs. competitive production
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_f3_value AS

    WITH player_ranks AS (
        SELECT
            p.*,

            -- Rank playing time within each franchise-season.
            -- Rank 1 = highest minutes per game.
            RANK() OVER (
                PARTITION BY franchise_id, season
                ORDER BY minutes_per_game DESC NULLS LAST
            ) AS team_minutes_rank

        FROM a02_f2_production AS p
    )

    SELECT
        p.*,

        -- 1–3: Salary allocation minus production shares.
        -- These are differences in proportions, not dollars.
        salary_share_recorded_roster - team_scoring_share
            AS salary_scoring_share_gap,

        salary_share_recorded_roster - team_assist_share
            AS salary_assist_share_gap,

        salary_share_recorded_roster - team_rebound_share
            AS salary_rebound_share_gap,

        -- 4: Salary hierarchy versus scoring hierarchy.
        salary_rank_franchise_season - team_points_rank
            AS salary_scoring_rank_gap,

        -- 5–6: Relative allocation ratios.
        -- NULLIF avoids division by zero.
        salary_share_recorded_roster
            / NULLIF(team_scoring_share, 0)
            AS salary_to_scoring_share_ratio,

        salary_share_recorded_roster
            / NULLIF(team_minutes_share, 0)
            AS salary_to_minutes_share_ratio,

        -- 7–8: Salary hierarchy versus playing time and usage.
        salary_rank_franchise_season - team_minutes_rank
            AS salary_minutes_rank_gap,

        salary_rank_franchise_season - team_usage_rank
            AS salary_usage_rank_gap

    FROM player_ranks AS p
""")

print("A02 F3 compensation-production view created.")


A02 F3 compensation-production view created.


In [24]:

# ------------------------------------------------------------
# A02 F3 — Grain and feature coverage
# ------------------------------------------------------------

f3_check = con.execute("""
    SELECT
        COUNT(*) AS rows,

        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,

        COUNT(salary) AS salary_matched,

        COUNT(salary_scoring_share_gap)
            AS scoring_gap_available,

        COUNT(salary_assist_share_gap)
            AS assist_gap_available,

        COUNT(salary_rebound_share_gap)
            AS rebound_gap_available,

        COUNT(salary_scoring_rank_gap)
            AS scoring_rank_gap_available,

        COUNT(salary_to_scoring_share_ratio)
            AS scoring_ratio_available,

        COUNT(salary_to_minutes_share_ratio)
            AS minutes_ratio_available,

        COUNT(salary_minutes_rank_gap)
            AS minutes_rank_gap_available,

        COUNT(salary_usage_rank_gap)
            AS usage_rank_gap_available

    FROM a02_f3_value
""").df()

display(f3_check)


,rows,unique_keys,salary_matched,scoring_gap_available,assist_gap_available,rebound_gap_available,scoring_rank_gap_available,scoring_ratio_available,minutes_ratio_available,minutes_rank_gap_available,usage_rank_gap_available
0,12810,12810,10601,10601,10601,10601,10601,10558,10601,10601,10601


In [25]:

from pathlib import Path

# Show the directory from which Jupyter resolves relative file paths.
print("Current working directory:", Path.cwd())

# Check the two most likely locations of the canonical CSV.
for path in [
    Path("data/canonical/player_awards.csv"),
    Path("../data/canonical/player_awards.csv")
]:
    print(f"{path} -> Exists: {path.exists()}")


Current working directory: /Users/jonathantran/Desktop/nba_economics/notebooks
data/canonical/player_awards.csv -> Exists: False
../data/canonical/player_awards.csv -> Exists: True


In [26]:

# ------------------------------------------------------------
# F4 — Register canonical C08 player awards
# ------------------------------------------------------------

from pathlib import Path

# Resolve the CSV path relative to the notebooks directory.
awards_path = Path("../data/canonical/player_awards.csv").resolve()

# Confirm that the file exists before querying it.
assert awards_path.is_file(), f"File not found: {awards_path}"

# Insert the verified file path directly into the SQL.
# DuckDB reads the CSV without modifying the canonical file.
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW c08_player_awards AS
    SELECT *
    FROM read_csv_auto(
        '{awards_path.as_posix()}',
        header = true
    )
""")

print("C08 registered successfully.")

# Inspect the actual columns and their data types.
display(con.execute("""
    DESCRIBE c08_player_awards
""").df())


C08 registered successfully.


,column_name,column_type,null,key,default,extra
0,season,VARCHAR,YES,None,None,None
1,source_player_name,VARCHAR,YES,None,None,None
2,player_position,VARCHAR,YES,None,None,None
3,player_id,BIGINT,YES,None,None,None
4,award_type,VARCHAR,YES,None,None,None
5,source_table,VARCHAR,YES,None,None,None
6,team_selection,VARCHAR,YES,None,None,None
7,age,DOUBLE,YES,None,None,None
8,team,VARCHAR,YES,None,None,None
9,points_won,DOUBLE,YES,None,None,None


In [27]:

# ------------------------------------------------------------
# F4 — C08 award categories and voting coverage
# ------------------------------------------------------------

c08_award_audit = con.execute("""
    SELECT
        award_type,

        COUNT(*) AS rows,

        COUNT(DISTINCT (player_id, season))
            AS unique_player_seasons,

        COUNT(voting_rank) AS voting_rank_available,

        COUNT(vote_share) AS vote_share_available,

        COUNT(points_won) AS voting_points_available,

        COUNT(team_selection) AS team_selection_available,

        COUNT(award_winner) AS winner_flag_available,

        COUNT(*) FILTER (
            WHERE award_winner = TRUE
        ) AS winner_true_count,

        MIN(voting_rank) AS min_voting_rank,
        MAX(voting_rank) AS max_voting_rank

    FROM c08_player_awards

    GROUP BY award_type
    ORDER BY award_type
""").df()

display(c08_award_audit)


,award_type,rows,unique_player_seasons,voting_rank_available,vote_share_available,voting_points_available,team_selection_available,winner_flag_available,winner_true_count,min_voting_rank,max_voting_rank
0,all_defensive,263,263,0,263,263,263,0,0,NaN,NaN
1,all_nba,390,390,0,0,0,390,0,0,NaN,NaN
2,all_rookie,260,260,0,260,260,260,0,0,NaN,NaN
3,all_star,681,681,0,0,0,0,0,0,NaN,NaN
4,dpoy,426,426,426,426,426,0,426,26,1.0,25.0
5,mvp,352,352,352,352,352,0,352,26,1.0,17.0
6,roy,198,198,198,198,198,0,198,26,1.0,13.0


In [28]:

# ------------------------------------------------------------
# F4 — Inspect All-Star selections in canonical C08
# ------------------------------------------------------------

display(con.execute("""
    SELECT
        season,
        source_player_name,
        player_id,
        team,
        selection_type,
        player_status,
        all_star_appearances
    FROM c08_player_awards
    WHERE award_type = 'all_star'
    ORDER BY season DESC, source_player_name
    LIMIT 25
""").df())

# Check the number of All-Star records in each season.
display(con.execute("""
    SELECT
        season,
        COUNT(*) AS all_star_records,
        COUNT(DISTINCT player_id) AS unique_players,
        COUNT(*) FILTER (
            WHERE player_id IS NULL
        ) AS missing_player_ids
    FROM c08_player_awards
    WHERE award_type = 'all_star'
    GROUP BY season
    ORDER BY season
""").df())


,season,source_player_name,player_id,team,selection_type,player_status,all_star_appearances
0,2025-2026,Alperen Şengün,1630578,Houston Rockets,reserve,REP,2.0
1,2025-2026,Anthony Edwards,1630162,Minnesota Timberwolves,reserve,NaN,4.0
2,2025-2026,Brandon Ingram,1627742,Toronto Raptors,reserve,REP,2.0
3,2025-2026,Cade Cunningham,1630595,Detroit Pistons,starter,NaN,2.0
4,2025-2026,Chet Holmgren,1631096,Oklahoma City Thunder,reserve,NaN,1.0
5,2025-2026,De'Aaron Fox,1628368,San Antonio Spurs,reserve,REP,2.0
6,2025-2026,Deni Avdija,1630166,Portland Trail Blazers,reserve,NaN,1.0
7,2025-2026,Devin Booker,1626164,Phoenix Suns,reserve,NaN,5.0
8,2025-2026,Donovan Mitchell,1628378,Cleveland Cavaliers,reserve,NaN,7.0
9,2025-2026,Giannis Antetokounmpo,203507,Milwaukee Bucks,starter,INJ,10.0


,season,all_star_records,unique_players,missing_player_ids
0,2000-2001,28,28,0
1,2001-2002,26,26,0
2,2002-2003,25,25,0
3,2003-2004,24,24,0
4,2004-2005,24,24,0
5,2005-2006,25,25,0
6,2006-2007,29,29,0
7,2007-2008,26,26,0
8,2008-2009,26,26,0
9,2009-2010,28,28,0


In [29]:

# ------------------------------------------------------------
# F4 — Final C08 pre-aggregation inspection
# ------------------------------------------------------------

# 1. Inspect All-NBA, All-Defensive, and All-Rookie
#    selection labels to determine how to encode them.
display(con.execute("""
    SELECT
        award_type,
        team_selection,
        COUNT(*) AS records
    FROM c08_player_awards
    WHERE award_type IN (
        'all_nba',
        'all_defensive',
        'all_rookie'
    )
    GROUP BY award_type, team_selection
    ORDER BY award_type, team_selection
""").df())

# 2. Check coverage and missing player IDs by award type.
#    Missing IDs would prevent a reliable player-level join.
display(con.execute("""
    SELECT
        award_type,
        COUNT(DISTINCT season) AS seasons_covered,
        MIN(season) AS first_season,
        MAX(season) AS last_season,
        COUNT(*) FILTER (
            WHERE player_id IS NULL
        ) AS missing_player_ids
    FROM c08_player_awards
    GROUP BY award_type
    ORDER BY award_type
""").df())

# 3. Inspect the All-Star selection/status combinations.
#    This will help us distinguish starters, reserves,
#    injury replacements, and other recorded statuses.
display(con.execute("""
    SELECT
        selection_type,
        player_status,
        COUNT(*) AS records
    FROM c08_player_awards
    WHERE award_type = 'all_star'
    GROUP BY selection_type, player_status
    ORDER BY selection_type, player_status
""").df())


,award_type,team_selection,records
0,all_defensive,1st,132
1,all_defensive,2nd,131
2,all_nba,1st,130
3,all_nba,2nd,130
4,all_nba,3rd,130
5,all_rookie,1st,131
6,all_rookie,2nd,129


,award_type,seasons_covered,first_season,last_season,missing_player_ids
0,all_defensive,26,2000-2001,2025-2026,0
1,all_nba,26,2000-2001,2025-2026,0
2,all_rookie,26,2000-2001,2025-2026,0
3,all_star,26,2000-2001,2025-2026,0
4,dpoy,26,2000-2001,2025-2026,0
5,mvp,26,2000-2001,2025-2026,0
6,roy,26,2000-2001,2025-2026,0


,selection_type,player_status,records
0,reserve,INJ,22
1,reserve,REP,45
2,reserve,SPL,2
3,reserve,NaN,352
4,starter,INJ,25
5,starter,NaN,235


In [30]:

# ------------------------------------------------------------
# F4 — Aggregate canonical awards to player-season grain
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f4_awards_player_season AS

SELECT
    player_id,
    season,

    -- Actual award wins and official team selections.
    MAX(CASE WHEN award_type = 'all_star'
        THEN 1 ELSE 0 END) AS is_all_star,

    MAX(CASE WHEN award_type = 'all_nba'
        THEN 1 ELSE 0 END) AS is_all_nba,

    MAX(CASE WHEN award_type = 'all_defensive'
        THEN 1 ELSE 0 END) AS is_all_defensive,

    MAX(CASE WHEN award_type = 'all_rookie'
        THEN 1 ELSE 0 END) AS is_all_rookie,

    MAX(CASE WHEN award_type = 'mvp'
                  AND award_winner = TRUE
        THEN 1 ELSE 0 END) AS is_mvp,

    MAX(CASE WHEN award_type = 'dpoy'
                  AND award_winner = TRUE
        THEN 1 ELSE 0 END) AS is_dpoy,

    MAX(CASE WHEN award_type = 'roy'
                  AND award_winner = TRUE
        THEN 1 ELSE 0 END) AS is_roy,

    -- Voting rank: NULL when no recorded voting position.
    MIN(CASE WHEN award_type = 'mvp'
        THEN voting_rank END) AS mvp_vote_rank,

    MIN(CASE WHEN award_type = 'dpoy'
        THEN voting_rank END) AS dpoy_vote_rank,

    MIN(CASE WHEN award_type = 'roy'
        THEN voting_rank END) AS roy_vote_rank,

    -- Top-three and top-five voting recognition.
    MAX(CASE WHEN award_type = 'mvp'
                  AND voting_rank <= 3
        THEN 1 ELSE 0 END) AS mvp_top_3,

    MAX(CASE WHEN award_type = 'mvp'
                  AND voting_rank <= 5
        THEN 1 ELSE 0 END) AS mvp_top_5,

    MAX(CASE WHEN award_type = 'dpoy'
                  AND voting_rank <= 3
        THEN 1 ELSE 0 END) AS dpoy_top_3,

    MAX(CASE WHEN award_type = 'dpoy'
                  AND voting_rank <= 5
        THEN 1 ELSE 0 END) AS dpoy_top_5,

    MAX(CASE WHEN award_type = 'roy'
                  AND voting_rank <= 3
        THEN 1 ELSE 0 END) AS roy_top_3,

    MAX(CASE WHEN award_type = 'roy'
                  AND voting_rank <= 5
        THEN 1 ELSE 0 END) AS roy_top_5,

    -- Preserve voting intensity, not only ordinal rank.
    MAX(CASE WHEN award_type = 'mvp'
        THEN vote_share END) AS mvp_vote_share,

    MAX(CASE WHEN award_type = 'dpoy'
        THEN vote_share END) AS dpoy_vote_share,

    MAX(CASE WHEN award_type = 'roy'
        THEN vote_share END) AS roy_vote_share,

    -- Official team designation: 1st, 2nd, or 3rd.
    MAX(CASE WHEN award_type = 'all_nba'
        THEN team_selection END) AS all_nba_team_selection,

    MAX(CASE WHEN award_type = 'all_defensive'
        THEN team_selection END) AS all_defensive_team_selection,

    MAX(CASE WHEN award_type = 'all_rookie'
        THEN team_selection END) AS all_rookie_team_selection,

    -- All-Star selection details.
    MAX(CASE WHEN award_type = 'all_star'
                  AND selection_type = 'starter'
        THEN 1 ELSE 0 END) AS is_all_star_starter,

    MAX(CASE WHEN award_type = 'all_star'
                  AND player_status = 'REP'
        THEN 1 ELSE 0 END) AS is_all_star_replacement,

    MAX(CASE WHEN award_type = 'all_star'
        THEN all_star_appearances END) AS all_star_appearances

FROM c08_player_awards
GROUP BY player_id, season
""")

print("F4 player-season awards aggregation created.")


F4 player-season awards aggregation created.


In [31]:

# ------------------------------------------------------------
# F4 — Validate awards aggregation
# ------------------------------------------------------------

display(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, season)) AS unique_keys,

        SUM(is_all_star) AS all_star_selections,
        SUM(is_all_nba) AS all_nba_selections,
        SUM(is_all_defensive) AS all_defensive_selections,

        SUM(is_mvp) AS mvp_winners,
        SUM(is_dpoy) AS dpoy_winners,
        SUM(is_roy) AS roy_winners,

        SUM(mvp_top_3) AS mvp_top_3_count,
        SUM(dpoy_top_3) AS dpoy_top_3_count,
        SUM(roy_top_3) AS roy_top_3_count

    FROM a02_f4_awards_player_season
""").df())


,rows,unique_keys,all_star_selections,all_nba_selections,all_defensive_selections,mvp_winners,dpoy_winners,roy_winners,mvp_top_3_count,dpoy_top_3_count,roy_top_3_count
0,1253,1253,681.0,390.0,263.0,26.0,26.0,26.0,78.0,79.0,80.0


In [32]:

# ------------------------------------------------------------
# F4 — Join player-season awards onto A02
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f4_awards AS

WITH awards_joined AS (
    SELECT
        a.*,

        -- Bring in all 25 award features from the
        -- player-season aggregation.
        w.* EXCLUDE (player_id, season)

    FROM a02_f3_value AS a

    LEFT JOIN a02_f4_awards_player_season AS w
        ON a.player_id = w.player_id
       AND a.season = w.season
)

SELECT
    *,

    -- Count actual awards/selections, not voting
    -- participation alone. COALESCE treats absent
    -- award records as zero for covered seasons.
    COALESCE(is_all_star, 0)
    + COALESCE(is_all_nba, 0)
    + COALESCE(is_all_defensive, 0)
    + COALESCE(is_all_rookie, 0)
    + COALESCE(is_mvp, 0)
    + COALESCE(is_dpoy, 0)
    + COALESCE(is_roy, 0)
        AS award_type_count

FROM awards_joined
""")

print("F4 awards joined to A02.")


F4 awards joined to A02.


In [33]:

# ------------------------------------------------------------
# F4 — Validate A02 row preservation and award coverage
# ------------------------------------------------------------

display(con.execute("""
    SELECT
        COUNT(*) AS rows,

        COUNT(DISTINCT (
            player_id,
            franchise_id,
            season
        )) AS unique_keys,

        COUNT(*) FILTER (
            WHERE is_all_star = 1
        ) AS all_star_rows,

        COUNT(*) FILTER (
            WHERE is_mvp = 1
        ) AS mvp_winner_rows,

        COUNT(*) FILTER (
            WHERE mvp_vote_rank IS NOT NULL
        ) AS mvp_voting_rows,

        COUNT(*) FILTER (
            WHERE award_type_count > 0
        ) AS player_franchise_rows_with_awards

    FROM a02_f4_awards
""").df())


,rows,unique_keys,all_star_rows,mvp_winner_rows,mvp_voting_rows,player_franchise_rows_with_awards
0,12810,12810,681,26,352,1085


In [34]:

# ------------------------------------------------------------
# F4 — Normalize award indicators after the LEFT JOIN
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f4_awards_final AS

SELECT
    * EXCLUDE (
        is_all_star,
        is_all_nba,
        is_all_defensive,
        is_all_rookie,
        is_mvp,
        is_dpoy,
        is_roy,
        mvp_top_3,
        mvp_top_5,
        dpoy_top_3,
        dpoy_top_5,
        roy_top_3,
        roy_top_5,
        is_all_star_starter,
        is_all_star_replacement
    ),

    -- Official award selections and wins.
    COALESCE(is_all_star, 0) AS is_all_star,
    COALESCE(is_all_nba, 0) AS is_all_nba,
    COALESCE(is_all_defensive, 0) AS is_all_defensive,
    COALESCE(is_all_rookie, 0) AS is_all_rookie,
    COALESCE(is_mvp, 0) AS is_mvp,
    COALESCE(is_dpoy, 0) AS is_dpoy,
    COALESCE(is_roy, 0) AS is_roy,

    -- Voting competitiveness indicators.
    COALESCE(mvp_top_3, 0) AS mvp_top_3,
    COALESCE(mvp_top_5, 0) AS mvp_top_5,
    COALESCE(dpoy_top_3, 0) AS dpoy_top_3,
    COALESCE(dpoy_top_5, 0) AS dpoy_top_5,
    COALESCE(roy_top_3, 0) AS roy_top_3,
    COALESCE(roy_top_5, 0) AS roy_top_5,

    -- All-Star distinctions.
    COALESCE(is_all_star_starter, 0)
        AS is_all_star_starter,

    COALESCE(is_all_star_replacement, 0)
        AS is_all_star_replacement

FROM a02_f4_awards
""")

print("F4 awards indicators normalized.")


F4 awards indicators normalized.


In [35]:

# ------------------------------------------------------------
# F4 — Final awards feature validation
# ------------------------------------------------------------

display(con.execute("""
SELECT
    COUNT(*) AS rows,

    COUNT(DISTINCT (
        player_id, franchise_id, season
    )) AS unique_keys,

    COUNT(*) FILTER (
        WHERE is_all_star IS NULL
           OR is_mvp IS NULL
           OR is_dpoy IS NULL
           OR is_roy IS NULL
    ) AS null_core_award_flags,

    COUNT(*) FILTER (
        WHERE mvp_top_3 = 1 AND mvp_top_5 <> 1
    ) AS invalid_mvp_top3,

    COUNT(*) FILTER (
        WHERE dpoy_top_3 = 1 AND dpoy_top_5 <> 1
    ) AS invalid_dpoy_top3,

    COUNT(*) FILTER (
        WHERE roy_top_3 = 1 AND roy_top_5 <> 1
    ) AS invalid_roy_top3,

    COUNT(*) FILTER (
        WHERE is_all_star_starter = 1
          AND is_all_star <> 1
    ) AS invalid_all_star_starter,

    COUNT(*) FILTER (
        WHERE is_all_star_replacement = 1
          AND is_all_star <> 1
    ) AS invalid_all_star_replacement,

    COUNT(*) FILTER (
        WHERE award_type_count < 0
           OR award_type_count > 7
    ) AS invalid_award_counts

FROM a02_f4_awards_final
""").df())


,rows,unique_keys,null_core_award_flags,invalid_mvp_top3,invalid_dpoy_top3,invalid_roy_top3,invalid_all_star_starter,invalid_all_star_replacement,invalid_award_counts
0,12810,12810,0,0,0,0,0,0,0


In [36]:

# Find the A01 views registered in our DuckDB connection.
display(con.execute("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_name ILIKE '%a01%'
    ORDER BY table_name
""").df())


,table_name,table_type
0,a01_reference,VIEW


In [37]:

# Inspect all columns in the registered A01 reference view.
a01_columns = con.execute("""
    DESCRIBE a01_reference
""").df()

print(f"Total A01 columns: {len(a01_columns)}")

# Display column names and data types.
display(a01_columns[["column_name", "column_type"]])


Total A01 columns: 210


,column_name,column_type
0,team_name,VARCHAR
1,wins,BIGINT
2,losses,BIGINT
3,win_pct,DOUBLE
4,games_behind,DOUBLE
...,...,...
205,lost_playoff_qualification,BOOLEAN
206,newly_reached_second_round,BOOLEAN
207,failed_to_repeat_second_round,BOOLEAN
208,newly_reached_conference_finals,BOOLEAN


In [38]:

# ------------------------------------------------------------
# F4 — Identify candidate A01 franchise-context features
# ------------------------------------------------------------

# Search for column names related to the four areas
# we want to bring into A02.
keywords = [
    "franchise_id", "season",
    "win", "loss", "playoff", "round",
    "payroll", "salary", "cap", "tax",
    "spend", "rank", "percentile",
    "prior", "prev", "lag", "yoy"
]

# Keep columns containing at least one keyword.
# The generator checks each keyword against the column name.
candidate_columns = a01_columns[
    a01_columns["column_name"].str.lower().apply(
        lambda name: any(word in name for word in keywords)
    )
]

print(f"Candidate columns: {len(candidate_columns)}")
print(candidate_columns.to_string(index=False))



Candidate columns: 135
                                  column_name column_type null  key default extra
                                         wins      BIGINT  YES None    None  None
                                       losses      BIGINT  YES None    None  None
                                      win_pct      DOUBLE  YES None    None  None
                                       season     VARCHAR  YES None    None  None
                                 franchise_id      BIGINT  YES None    None  None
                                  league_rank      BIGINT  YES None    None  None
                             pythagorean_wins      BIGINT  YES None    None  None
                           pythagorean_losses      BIGINT  YES None    None  None
                           playoff_series_won      BIGINT  YES None    None  None
                                  round_score      DOUBLE  YES None    None  None
                               furthest_round     VARCHAR  YES None    None

In [39]:

# ------------------------------------------------------------
# F4 — Join 15 A01 franchise-context features into A02
# ------------------------------------------------------------

# First, ensure A01 has one record per franchise-season.
# A duplicate here could multiply player rows during the join.
grain_check = con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (franchise_id, season)) AS unique_keys
    FROM a01_reference
""").fetchone()

assert grain_check[0] == grain_check[1], (
    f"A01 contains duplicate franchise-season keys: {grain_check}"
)

# Build the next analytical view without modifying
# our existing F4 awards view or the A01 reference.
con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f4_final AS

SELECT
    p.*,

    -- Franchise competitive outcomes.
    t.win_pct AS team_win_pct,
    t.made_playoffs AS team_made_playoffs,
    t.round_score AS team_round_score,
    t.win_pct_yoy_change AS team_win_pct_yoy_change,

    -- Franchise financial context.
    t.team_payroll AS context_team_payroll,
    t.payroll_to_cap_ratio AS team_payroll_to_cap_ratio,
    t.payroll_percentile AS team_payroll_percentile,
    t.team_payroll_yoy_growth AS team_payroll_yoy_growth,

    -- Luxury-tax context.
    t.above_tax_threshold AS team_above_tax_threshold,
    t.taxable_payroll_to_tax_ratio
        AS team_taxable_payroll_to_tax_ratio,
    t.tax_paid AS team_tax_paid,

    -- Franchise spending efficiency.
    t.payroll_normalized_efficiency
        AS team_payroll_normalized_efficiency,

    -- Multi-season competitive and financial context.
    t.avg_win_pct_3yr AS team_avg_win_pct_3yr,
    t.avg_payroll_to_cap_3yr AS team_avg_payroll_to_cap_3yr,
    t.playoff_appearances_3yr AS team_playoff_appearances_3yr

FROM a02_f4_awards_final AS p

LEFT JOIN a01_reference AS t
    ON p.franchise_id = t.franchise_id
   AND p.season = t.season
""")

print("F4 franchise context joined successfully.")


F4 franchise context joined successfully.


In [40]:

# ------------------------------------------------------------
# F4 — Validate final analytical layer
# ------------------------------------------------------------

display(con.execute("""
SELECT
    COUNT(*) AS rows,

    COUNT(DISTINCT (
        player_id, franchise_id, season
    )) AS unique_keys,

    COUNT(*) FILTER (
        WHERE team_win_pct IS NOT NULL
    ) AS rows_with_team_performance,

    COUNT(*) FILTER (
        WHERE context_team_payroll IS NOT NULL
    ) AS rows_with_team_payroll,

    COUNT(*) FILTER (
        WHERE team_payroll_to_cap_ratio IS NOT NULL
    ) AS rows_with_payroll_cap_ratio,

    COUNT(*) FILTER (
        WHERE team_tax_paid IS NOT NULL
    ) AS rows_with_tax_paid

FROM a02_f4_final
""").df())


,rows,unique_keys,rows_with_team_performance,rows_with_team_payroll,rows_with_payroll_cap_ratio,rows_with_tax_paid
0,12810,12810,12810,12810,12810,4040


In [41]:

# ------------------------------------------------------------
# A02 — Final dataset summary
# ------------------------------------------------------------

# Retrieve the schema so we can inspect the completed
# analytical layer without manually listing every feature.
final_schema = con.execute("""
    DESCRIBE a02_f4_final
""").df()

# Summarize size, unique keys, and coverage.
display(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(DISTINCT player_id) AS unique_players,
        COUNT(DISTINCT franchise_id) AS franchises,
        COUNT(DISTINCT season) AS seasons,
        COUNT(*) FILTER (
            WHERE salary IS NOT NULL
        ) AS salary_matched_rows,
        COUNT(*) FILTER (
            WHERE team_win_pct IS NOT NULL
        ) AS team_context_rows,
        COUNT(*) FILTER (
            WHERE award_type_count > 0
        ) AS rows_with_awards
    FROM a02_f4_final
""").df())

print(f"Final A02 columns: {len(final_schema)}")


,rows,unique_keys,unique_players,franchises,seasons,salary_matched_rows,team_context_rows,rows_with_awards
0,12810,12810,2584,30,26,10601,12810,1085


Final A02 columns: 156


In [42]:

# ------------------------------------------------------------
# F5.1 — Inspect candidate source columns
# ------------------------------------------------------------

# Read the schema of our completed F4 analytical view.
f5_schema = con.execute("""
    DESCRIBE a02_f4_final
""").df()

# Search for the existing fields needed to construct
# historical baselines and longitudinal features.
keywords = [
    "player_id", "franchise_id", "season",
    "games", "minutes", "points", "ppg",
    "salary", "cap", "scoring_share",
    "salary_scoring_share_gap"
]

# Return matching columns without printing all 156.
f5_candidates = f5_schema[
    f5_schema["column_name"].str.lower().apply(
        lambda name: any(word in name for word in keywords)
    )
]

print(f"Relevant columns: {len(f5_candidates)}")

display(
    f5_candidates[
        ["column_name", "column_type"]
    ]
)


Relevant columns: 51


,column_name,column_type
0,player_id,BIGINT
1,franchise_id,BIGINT
2,season,VARCHAR
6,games_played,BIGINT
7,points_total,BIGINT
13,minutes_per_game,DOUBLE
31,points_per_game,DOUBLE
55,salary,BIGINT
56,salary_available,INTEGER
57,salary_cap,BIGINT


In [43]:

# ------------------------------------------------------------
# F5.2 — Identify player-seasons with multiple franchises
# ------------------------------------------------------------

display(con.execute("""
    SELECT
        player_id,
        season,
        COUNT(*) AS franchise_records,
        COUNT(DISTINCT franchise_id) AS distinct_franchises

    FROM a02_f4_final

    GROUP BY
        player_id,
        season

    HAVING COUNT(DISTINCT franchise_id) > 1

    ORDER BY
        season DESC,
        distinct_franchises DESC,
        player_id

    LIMIT 20
""").df())

# Quantify how frequently multiple-franchise seasons occur.
display(con.execute("""
    WITH player_seasons AS (
        SELECT
            player_id,
            season,
            COUNT(DISTINCT franchise_id) AS franchise_count
        FROM a02_f4_final
        GROUP BY player_id, season
    )

    SELECT
        COUNT(*) AS total_player_seasons,
        COUNT(*) FILTER (
            WHERE franchise_count > 1
        ) AS multi_franchise_player_seasons,
        MAX(franchise_count) AS max_franchises_in_season

    FROM player_seasons
""").df())


,player_id,season,franchise_records,distinct_franchises


,total_player_seasons,multi_franchise_player_seasons,max_franchises_in_season
0,12810,0,1


In [44]:

# ------------------------------------------------------------
# F5.1 — Temporal foundation
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f5_temporal_base AS

SELECT
    *,

    -- Convert '2023-24' into 2023 for chronological ordering.
    TRY_CAST(SUBSTR(season, 1, 4) AS INTEGER)
        AS season_start_year,

    -- Normalize scoring output to 36 minutes of playing time.
    -- NULLIF prevents division by zero.
    CASE
        WHEN minutes_per_game > 0
        THEN points_per_game * 36.0 / minutes_per_game
        ELSE NULL
    END AS points_per_36

FROM a02_f4_final
""")

print("F5 temporal foundation created.")

display(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, season)) AS unique_player_seasons,
        MIN(season_start_year) AS first_year,
        MAX(season_start_year) AS last_year
    FROM a02_f5_temporal_base
""").df())


F5 temporal foundation created.


,rows,unique_player_seasons,first_year,last_year
0,12810,12810,2000,2025


In [45]:

# ------------------------------------------------------------
# F5.2 — Previous observations and franchise tenure
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f5_lagged AS

WITH ordered AS (
    SELECT
        *,

        -- Previous observed season, which may not be consecutive.
        LAG(season_start_year) OVER w AS prior_observed_year,
        LAG(franchise_id) OVER w AS prior_observed_franchise,

        LAG(points_per_game) OVER w AS lag_ppg,
        LAG(minutes_per_game) OVER w AS lag_minutes,
        LAG(points_per_36) OVER w AS lag_points_per_36,
        LAG(salary) OVER w AS lag_salary,
        LAG(salary_share_recorded_roster) OVER w AS lag_salary_share,
        LAG(salary_to_cap_ratio) OVER w AS lag_salary_cap_ratio,
        LAG(salary_scoring_share_gap) OVER w AS lag_value_gap,

        -- First observed season in our 2000–2025 coverage.
        MIN(season_start_year) OVER (
            PARTITION BY player_id
        ) AS first_observed_year

    FROM a02_f5_temporal_base

    WINDOW w AS (
        PARTITION BY player_id
        ORDER BY season_start_year
    )
),

continuity AS (
    SELECT
        *,

        -- A new tenure starts for a first observation,
        -- a season gap, or a franchise change.
        CASE
            WHEN prior_observed_year IS NULL
              OR season_start_year - prior_observed_year <> 1
              OR franchise_id <> prior_observed_franchise
            THEN 1
            ELSE 0
        END AS new_franchise_stint

    FROM ordered
),

stints AS (
    SELECT
        *,

        -- Cumulative count of new stints creates a group ID
        -- within each player's history.
        SUM(new_franchise_stint) OVER (
            PARTITION BY player_id
            ORDER BY season_start_year
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS franchise_stint_id

    FROM continuity
)

SELECT
    *,

    -- Count seasons within the current uninterrupted stint.
    ROW_NUMBER() OVER (
        PARTITION BY player_id, franchise_stint_id
        ORDER BY season_start_year
    ) AS consecutive_seasons_with_franchise

FROM stints
""")

print("F5 previous-observation and tenure view created.")


F5 previous-observation and tenure view created.


In [46]:

# ------------------------------------------------------------
# F5.3 — Final longitudinal feature calculations
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f5_final AS

WITH history AS (
    SELECT
        b.*,

        -- Historical baselines use only preceding seasons.
        AVG(h.points_per_game) AS avg_ppg_prior_3yr,
        AVG(h.minutes_per_game) AS avg_minutes_prior_3yr,
        AVG(h.salary_share_recorded_roster)
            AS avg_salary_share_prior_3yr

    FROM a02_f5_lagged b

    LEFT JOIN a02_f5_temporal_base h
        ON b.player_id = h.player_id
       AND h.season_start_year
           BETWEEN b.season_start_year - 3
               AND b.season_start_year - 1

    GROUP BY ALL
),

features AS (
    SELECT
        *,

        -- An immediately preceding season must exist.
        (prior_observed_year = season_start_year - 1)
            AS has_previous_season,

        -- A franchise change is defined only when a
        -- consecutive previous-season record exists.
        CASE
            WHEN prior_observed_year = season_start_year - 1
            THEN franchise_id <> prior_observed_franchise
            ELSE NULL
        END AS changed_franchise,

        -- Years since the first observation in this dataset.
        season_start_year - first_observed_year
            AS seasons_since_first_observed,

        -- Group A: Previous-season baselines.
        CASE WHEN has_previous_season THEN lag_ppg END
            AS prev_ppg,

        CASE WHEN has_previous_season THEN lag_minutes END
            AS prev_minutes_per_game,

        CASE WHEN has_previous_season THEN lag_points_per_36 END
            AS prev_points_per_36,

        CASE WHEN has_previous_season THEN lag_salary END
            AS prev_salary,

        CASE WHEN has_previous_season THEN lag_salary_share END
            AS prev_salary_share,

        CASE WHEN has_previous_season THEN lag_salary_cap_ratio END
            AS prev_salary_to_cap_ratio,

        CASE WHEN has_previous_season THEN lag_value_gap END
            AS prev_salary_scoring_share_gap

    FROM history
)

SELECT
    *,

    -- Group B: Year-over-year absolute and relative changes.
    points_per_game - prev_ppg AS ppg_yoy_change,

    CASE
        WHEN prev_ppg > 0
        THEN (points_per_game - prev_ppg) / prev_ppg
        ELSE NULL
    END AS ppg_yoy_growth,

    minutes_per_game - prev_minutes_per_game
        AS minutes_yoy_change,

    points_per_36 - prev_points_per_36
        AS points_per_36_yoy_change,

    salary - prev_salary AS salary_yoy_change,

    CASE
        WHEN prev_salary > 0
        THEN (salary - prev_salary) * 1.0 / prev_salary
        ELSE NULL
    END AS salary_yoy_growth,

    salary_share_recorded_roster - prev_salary_share
        AS salary_share_yoy_change,

    salary_to_cap_ratio - prev_salary_to_cap_ratio
        AS salary_to_cap_yoy_change,

    salary_scoring_share_gap - prev_salary_scoring_share_gap
        AS value_gap_yoy_change,

    -- Group D: Current scoring versus historical baseline.
    points_per_game - avg_ppg_prior_3yr
        AS ppg_vs_prior_3yr

FROM features
""")

print("F5 longitudinal features created.")


F5 longitudinal features created.


In [47]:

# ------------------------------------------------------------
# F5.4 — Initial structural and temporal checks
# ------------------------------------------------------------

display(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,

        COUNT(*) FILTER (
            WHERE has_previous_season
        ) AS consecutive_player_seasons,

        COUNT(*) FILTER (
            WHERE changed_franchise
        ) AS franchise_changes,

        COUNT(*) FILTER (
            WHERE prev_ppg IS NOT NULL
        ) AS previous_ppg_coverage,

        COUNT(*) FILTER (
            WHERE prev_salary IS NOT NULL
        ) AS previous_salary_coverage,

        COUNT(*) FILTER (
            WHERE avg_ppg_prior_3yr IS NOT NULL
        ) AS rolling_ppg_coverage,

        MIN(consecutive_seasons_with_franchise)
            AS minimum_franchise_tenure,

        MAX(consecutive_seasons_with_franchise)
            AS maximum_franchise_tenure

    FROM a02_f5_final
""").df())


,rows,unique_keys,consecutive_player_seasons,franchise_changes,previous_ppg_coverage,previous_salary_coverage,rolling_ppg_coverage,minimum_franchise_tenure,maximum_franchise_tenure
0,12810,12810,9850,3952,9850,8620,10179,1,20


In [48]:

# ------------------------------------------------------------
# F5.5 — Targeted temporal and semantic validation
# ------------------------------------------------------------

f5_audit = con.execute("""
WITH checks AS (
    SELECT
        COUNT(*) AS total_rows,

        -- Prior-season features must not exist without
        -- an immediately preceding season.
        COUNT(*) FILTER (
            WHERE NOT COALESCE(has_previous_season, FALSE)
              AND prev_ppg IS NOT NULL
        ) AS invalid_previous_ppg,

        -- Previous-season PPG should match the lagged value.
        COUNT(*) FILTER (
            WHERE has_previous_season
              AND prev_ppg IS DISTINCT FROM lag_ppg
        ) AS previous_ppg_mismatches,

        -- Franchise changes must be based on actual changes
        -- between consecutive player-season observations.
        COUNT(*) FILTER (
            WHERE has_previous_season
              AND changed_franchise IS DISTINCT FROM
                  (franchise_id <> prior_observed_franchise)
        ) AS franchise_change_mismatches,

        -- A new franchise stint must start at tenure 1.
        COUNT(*) FILTER (
            WHERE new_franchise_stint = 1
              AND consecutive_seasons_with_franchise <> 1
        ) AS invalid_stint_starts,

        -- Salary history must remain NULL when the prior
        -- salary observation is unavailable.
        COUNT(*) FILTER (
            WHERE has_previous_season
              AND lag_salary IS NULL
              AND prev_salary IS NOT NULL
        ) AS invalid_salary_imputation,

        -- Three-year averages must only use historical rows.
        COUNT(*) FILTER (
            WHERE avg_ppg_prior_3yr IS NOT NULL
              AND first_observed_year >= season_start_year
        ) AS invalid_rolling_history

    FROM a02_f5_final
)
SELECT * FROM checks
""").df()

display(f5_audit)


,total_rows,invalid_previous_ppg,previous_ppg_mismatches,franchise_change_mismatches,invalid_stint_starts,invalid_salary_imputation,invalid_rolling_history
0,12810,0,0,0,0,0,0


In [49]:

# ------------------------------------------------------------
# F5.6 — Final clean A02 view
# ------------------------------------------------------------

# List only the F5 features we want to retain.
f5_features = [
    "season_start_year",
    "points_per_36",
    "has_previous_season",
    "changed_franchise",
    "consecutive_seasons_with_franchise",
    "seasons_since_first_observed",
    "prev_ppg",
    "prev_minutes_per_game",
    "prev_points_per_36",
    "prev_salary",
    "prev_salary_share",
    "prev_salary_to_cap_ratio",
    "prev_salary_scoring_share_gap",
    "ppg_yoy_change",
    "ppg_yoy_growth",
    "minutes_yoy_change",
    "points_per_36_yoy_change",
    "salary_yoy_change",
    "salary_yoy_growth",
    "salary_share_yoy_change",
    "salary_to_cap_yoy_change",
    "value_gap_yoy_change",
    "avg_ppg_prior_3yr",
    "avg_minutes_prior_3yr",
    "avg_salary_share_prior_3yr",
    "ppg_vs_prior_3yr",
]

# Retrieve original columns so we can preserve the F4 schema.
original_cols = con.execute("""
    DESCRIBE a02_f4_final
""").df()["column_name"].tolist()

# Build a comma-separated SQL projection using the exact
# column names rather than SELECT * from the helper view.
final_columns = original_cols + f5_features
select_columns = ",\n    ".join(
    f'"{column}"' for column in final_columns
)

con.execute(f"""
CREATE OR REPLACE TEMP VIEW a02_f5_clean AS
SELECT
    {select_columns}
FROM a02_f5_final
""")

display(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys
    FROM a02_f5_clean
""").df())

print(f"Final A02 columns: {len(final_columns)}")


,rows,unique_keys
0,12810,12810


Final A02 columns: 182


In [50]:

# ------------------------------------------------------------
# F5.7 — Historical observation coverage
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f5_history_coverage AS

SELECT
    current_row.player_id,
    current_row.season,

    -- Count observed seasons in the preceding 3 calendar years.
    COUNT(history.player_id) AS prior_3yr_seasons_observed

FROM a02_f5_temporal_base current_row

LEFT JOIN a02_f5_temporal_base history
    ON current_row.player_id = history.player_id
   AND history.season_start_year
       BETWEEN current_row.season_start_year - 3
           AND current_row.season_start_year - 1

GROUP BY
    current_row.player_id,
    current_row.season
""")

con.execute("""
CREATE OR REPLACE TEMP VIEW a02_f5_ready AS

SELECT
    f.*,
    h.prior_3yr_seasons_observed

FROM a02_f5_clean f

LEFT JOIN a02_f5_history_coverage h
    ON f.player_id = h.player_id
   AND f.season = h.season
""")

display(con.execute("""
    SELECT
        prior_3yr_seasons_observed,
        COUNT(*) AS player_seasons

    FROM a02_f5_ready

    GROUP BY 1
    ORDER BY 1
""").df())


,prior_3yr_seasons_observed,player_seasons
0,0,2631
1,1,2084
2,2,1840
3,3,6255


In [51]:

# ------------------------------------------------------------
# F5.8 — Final F5 integrity checks
# ------------------------------------------------------------

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT (player_id, franchise_id, season))
        AS unique_keys,

    COUNT(*) FILTER (
        WHERE prior_3yr_seasons_observed NOT BETWEEN 0 AND 3
    ) AS invalid_history_counts,

    COUNT(*) FILTER (
        WHERE prior_3yr_seasons_observed = 0
          AND avg_ppg_prior_3yr IS NOT NULL
    ) AS invalid_empty_history,

    COUNT(*) FILTER (
        WHERE prior_3yr_seasons_observed > 0
          AND avg_ppg_prior_3yr IS NULL
    ) AS missing_historical_ppg,

    COUNT(*) FILTER (
        WHERE has_previous_season
          AND prev_ppg IS NULL
    ) AS missing_previous_ppg

FROM a02_f5_ready
""").df())

print(
    "Final F5 columns:",
    len(con.execute("DESCRIBE a02_f5_ready").df())
)


,rows,unique_keys,invalid_history_counts,invalid_empty_history,missing_historical_ppg,missing_previous_ppg
0,12810,12810,0,0,0,0


Final F5 columns: 183


In [52]:

# ============================================================
# A02 QA CLOSEOUT — CHECK 1
# Final structural integrity and F4-to-F5 reconciliation
# ============================================================

# 1. Confirm the final view has the expected grain.
summary = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (player_id, season)) AS unique_player_seasons,
    COUNT(DISTINCT (player_id, franchise_id, season)) AS unique_keys,
    COUNT(*) FILTER (
        WHERE player_id IS NULL
           OR franchise_id IS NULL
           OR season IS NULL
    ) AS null_keys,
    COUNT(DISTINCT season) AS seasons,
    COUNT(DISTINCT player_id) AS players,
    COUNT(DISTINCT franchise_id) AS franchises
FROM a02_f5_ready
""").df()

display(summary)

# 2. Compare all original F4 columns against the final F5 view.
# EXCEPT detects rows whose inherited values differ.
# This checks more than just key preservation.

f4_columns = con.execute(
    "DESCRIBE a02_f4_final"
).df()["column_name"].tolist()

projection = ", ".join(
    f'"{column}"' for column in f4_columns
)

reconciliation = con.execute(f"""
WITH
missing_or_changed AS (
    SELECT {projection} FROM a02_f4_final
    EXCEPT
    SELECT {projection} FROM a02_f5_ready
),
unexpected_or_changed AS (
    SELECT {projection} FROM a02_f5_ready
    EXCEPT
    SELECT {projection} FROM a02_f4_final
)
SELECT
    (SELECT COUNT(*) FROM missing_or_changed)
        AS missing_or_changed_f4_rows,
    (SELECT COUNT(*) FROM unexpected_or_changed)
        AS unexpected_or_changed_f5_rows
""").df()

display(reconciliation)

# 3. Confirm the final schema.
schema = con.execute(
    "DESCRIBE a02_f5_ready"
).df()

print("Final columns:", len(schema))
print(
    "Duplicate column names:",
    len(schema) - schema["column_name"].nunique()
)


,total_rows,unique_player_seasons,unique_keys,null_keys,seasons,players,franchises
0,12810,12810,12810,0,26,2584,30


,missing_or_changed_f4_rows,unexpected_or_changed_f5_rows
0,665,593


Final columns: 183
Duplicate column names: 0


In [53]:

# Identify shared floating-point columns in F4 and F5.
f4_schema = con.execute("DESCRIBE a02_f4_final").df()
f5_schema = con.execute("DESCRIBE a02_f5_ready").df()

f4_float_cols = set(
    f4_schema.loc[
        f4_schema["column_type"].isin(["FLOAT", "DOUBLE"]),
        "column_name"
    ]
)

f5_float_cols = set(
    f5_schema.loc[
        f5_schema["column_type"].isin(["FLOAT", "DOUBLE"]),
        "column_name"
    ]
)

# Compare all shared floating-point columns.
affected_cols = sorted(f4_float_cols & f5_float_cols)

print("Floating-point columns checked:", len(affected_cols))




# Final tolerance-based reconciliation for the seven columns.
# A mismatch only matters if the null status differs or the
# relative numerical difference exceeds 1e-10.

checks = []

for col in affected_cols:
    q = '"' + col.replace('"', '""') + '"'

    checks.append(f"""
    COUNT(*) FILTER (
        WHERE (f4.{q} IS NULL) != (f5.{q} IS NULL)
           OR (
               f4.{q} IS NOT NULL
               AND f5.{q} IS NOT NULL
               AND ABS(f4.{q} - f5.{q}) >
                   1e-10 * GREATEST(
                       1.0,
                       ABS(f4.{q}),
                       ABS(f5.{q})
                   )
           )
    ) AS {q}
    """)

sql = f"""
SELECT {", ".join(checks)}
FROM a02_f4_final f4
JOIN a02_f5_ready f5
  ON f4.player_id = f5.player_id
 AND f4.franchise_id = f5.franchise_id
 AND f4.season = f5.season
"""

tolerance_results = con.execute(sql).df()

display(tolerance_results.T.rename(columns={0: "material_mismatches"}))


Floating-point columns checked: 96


,material_mismatches
age,0
all_star_appearances,0
assist_pct,0
assist_ratio,0
assist_to_turnover_ratio,0
...,...
three_pointers_made_per_game,0
true_shooting_pct,0
turnovers_per_game,0
usage_pct,0


In [54]:

# ============================================================
# A02 QA CLOSEOUT — CHECK 04A
# Coverage and missingness reconciliation
# ============================================================

# Inspect the final view's schema.
schema = con.execute("DESCRIBE a02_f5_ready").df()
available = set(schema["column_name"])

# Expected coverage fields from the A02 feature families.
fields = {
    "salary_matched": "salary",
    "team_win_pct_available": "team_win_pct",
    "team_tax_paid_available": "team_tax_paid",
    "previous_season_available": "has_previous_season",
    "previous_salary_available": "prev_salary",
    "prior_3yr_ppg_available": "avg_ppg_prior_3yr"
}

expressions = ["COUNT(*) AS total_rows"]
missing_fields = []

for metric, column in fields.items():

    # Report absent fields instead of guessing their names.
    if column not in available:
        missing_fields.append(column)
        continue

    # A flag must be counted by its TRUE value.
    if column == "has_previous_season":
        expressions.append(
            f'COUNT(*) FILTER (WHERE "{column}" = 1) '
            f'AS "{metric}"'
        )
    else:
        # COUNT(column) excludes SQL NULL values.
        expressions.append(
            f'COUNT("{column}") AS "{metric}"'
        )

query = f"""
SELECT {", ".join(expressions)}
FROM a02_f5_ready
"""

coverage_report = con.execute(query).df()

display(
    coverage_report.T.rename(columns={0: "observed_count"})
)

print("Missing expected column names:", missing_fields)

# Inspect possible award columns if the precise award flag
# was not already identified in our coverage mapping.
award_columns = [
    c for c in schema["column_name"]
    if "award" in c.lower()
    or "all_star" in c.lower()
    or "all_nba" in c.lower()
]

print("Candidate award columns:", award_columns)


,observed_count
total_rows,12810
salary_matched,10601
team_win_pct_available,12810
team_tax_paid_available,4040
previous_season_available,9850
previous_salary_available,8620
prior_3yr_ppg_available,10179


Missing expected column names: []
Candidate award columns: ['all_nba_team_selection', 'all_star_appearances', 'award_type_count', 'is_all_star', 'is_all_nba', 'is_all_star_starter', 'is_all_star_replacement']


In [55]:

# ============================================================
# A02 QA CLOSEOUT — CHECK 04B
# Historical observation-window coverage
# ============================================================

history_report = con.execute("""
SELECT
    prior_3yr_seasons_observed,
    COUNT(*) AS player_seasons,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        1
    ) AS pct_of_dataset
FROM a02_f5_ready
GROUP BY prior_3yr_seasons_observed
ORDER BY prior_3yr_seasons_observed
""").df()

display(history_report)


,prior_3yr_seasons_observed,player_seasons,pct_of_dataset
0,0,2631,20.5
1,1,2084,16.3
2,2,1840,14.4
3,3,6255,48.8


In [56]:

# ============================================================
# A02 QA CLOSEOUT — CHECK 05
# Award coverage and historical feature consistency
# ============================================================

semantic_check = con.execute("""
SELECT
    -- Number of player-season rows with recorded awards.
    COUNT(*) FILTER (
        WHERE award_type_count > 0
    ) AS rows_with_awards,

    -- Historical observation count should be between 0 and 3.
    COUNT(*) FILTER (
        WHERE prior_3yr_seasons_observed NOT BETWEEN 0 AND 3
           OR prior_3yr_seasons_observed IS NULL
    ) AS invalid_history_counts,

    -- A three-year historical average should not exist
    -- when no historical observations are available.
    COUNT(*) FILTER (
        WHERE prior_3yr_seasons_observed = 0
          AND avg_ppg_prior_3yr IS NOT NULL
    ) AS history_average_without_history,

    -- A historical average should exist when prior
    -- observed seasons contain usable PPG values.
    COUNT(*) FILTER (
        WHERE prior_3yr_seasons_observed > 0
          AND avg_ppg_prior_3yr IS NULL
    ) AS missing_historical_average,

    -- Previous-season PPG should be present when the
    -- immediately preceding season exists.
    COUNT(*) FILTER (
        WHERE has_previous_season = 1
          AND prev_ppg IS NULL
    ) AS missing_previous_ppg

FROM a02_f5_ready
""").df()

display(semantic_check.T.rename(columns={0: "observed_count"}))


,observed_count
rows_with_awards,1085
invalid_history_counts,0
history_average_without_history,0
missing_historical_average,0
missing_previous_ppg,0


In [57]:

from pathlib import Path
import duckdb

# Identify the repository root.
cwd = Path.cwd()
repo_root = cwd if (cwd / "notebooks").is_dir() else cwd.parent

# Define the final Parquet destination.
output_path = repo_root / "data/analytical/a02_player_franchise_season.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)





# ============================================================
# A02 QA — CHECK 06E
# Preserve integer types during Parquet export
# ============================================================

# These five totals are integer-valued aggregations.
integer_columns = {
    "matched_roster_salary_total",
    "matched_league_salary_total",
    "recorded_team_points_total",
    "recorded_team_assists_total",
    "recorded_team_rebounds_total"
}


# Read the schema directly from the final F5 analytical view.
# This avoids relying on variables from earlier QA cells.
source_schema = con.execute(
    "DESCRIBE a02_f5_ready"
).df()[["column_name", "column_type"]]

# Preserve the original column order and names.
all_columns = source_schema["column_name"].tolist()


select_expressions = []

for column in all_columns:
    quoted = '"' + column.replace('"', '""') + '"'

    if column in integer_columns:
        # BIGINT is stored as an integer in Parquet.
        select_expressions.append(
            f"CAST({quoted} AS BIGINT) AS {quoted}"
        )
    else:
        select_expressions.append(quoted)

select_clause = ",\n    ".join(select_expressions)

# Replace the existing export at its correct repository path.
con.execute(f"""
COPY (
    SELECT
        {select_clause}
    FROM a02_f5_ready
)
TO '{output_path.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD, OVERWRITE_OR_IGNORE TRUE)
""")

# Inspect the new Parquet schema.
verify_con = duckdb.connect()

new_export_schema = verify_con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(output_path)]
).df()[["column_name", "column_type"]]

display(
    new_export_schema[
        new_export_schema["column_name"].isin(integer_columns)
    ]
)

verify_con.close()


,column_name,column_type
74,matched_roster_salary_total,BIGINT
75,matched_league_salary_total,BIGINT
90,recorded_team_points_total,BIGINT
91,recorded_team_assists_total,BIGINT
92,recorded_team_rebounds_total,BIGINT


In [58]:

# ============================================================
# A02 QA — CHECK 06F (CORRECTED)
# Parquet value and schema integrity
# ============================================================

# DuckDB CREATE VIEW does not support prepared parameters.
# Escape single quotes so the path is valid inside SQL.
parquet_path_sql = str(output_path).replace("'", "''")

# Create a read-back view in the existing DuckDB connection.
# This allows a direct comparison with a02_f5_ready.
con.execute(f"""
CREATE OR REPLACE TEMP VIEW a02_export_check AS
SELECT *
FROM read_parquet('{parquet_path_sql}')
""")

integer_cols = [
    "matched_roster_salary_total",
    "matched_league_salary_total",
    "recorded_team_points_total",
    "recorded_team_assists_total",
    "recorded_team_rebounds_total"
]

# Compare the five integer totals, including NULL differences.
checks = []

for col in integer_cols:
    checks.append(f"""
        COUNT(*) FILTER (
            WHERE src."{col}" IS DISTINCT FROM exp."{col}"
        ) AS "{col}"
    """)

result = con.execute(f"""
SELECT
    {", ".join(checks)}
FROM a02_f5_ready src
JOIN a02_export_check exp
  ON src.player_id = exp.player_id
 AND src.franchise_id = exp.franchise_id
 AND src.season = exp.season
""").df()

print("Exported value mismatches (expected all zeros):")
display(result.T.rename(columns={0: "mismatches"}))

# Compare source and exported column names and types.
source_schema = con.execute(
    "DESCRIBE a02_f5_ready"
).df()[["column_name", "column_type"]]

export_schema = con.execute(
    "DESCRIBE a02_export_check"
).df()[["column_name", "column_type"]]

# Adjust only the five deliberately converted types.
expected_schema = source_schema.copy()

expected_schema.loc[
    expected_schema["column_name"].isin(integer_cols),
    "column_type"
] = "BIGINT"

print(
    "Schema matches expected export:",
    expected_schema.reset_index(drop=True).equals(
        export_schema.reset_index(drop=True)
    )
)

print("Exported columns:", len(export_schema))
print("Parquet path:", output_path)


Exported value mismatches (expected all zeros):


,mismatches
matched_roster_salary_total,0
matched_league_salary_total,0
recorded_team_points_total,0
recorded_team_assists_total,0
recorded_team_rebounds_total,0


Schema matches expected export: True
Exported columns: 183
Parquet path: /Users/jonathantran/Desktop/nba_economics/data/analytical/a02_player_franchise_season.parquet
